# Corpus A — preparação auditável de livros de linguística

Primeira obra: **A coerência textual**, Ingedore Villaça Koch e Luiz Carlos Travaglia.
Notebook completo, autocontido e reutilizável para uma obra por vez.
Execute as células em ordem.

Inclui PDF preservado, extração, limpeza reversível, segmentação estrutural,
auditoria e exportação. **Não executa E5, embeddings ou interpretação linguística.**

No Colab, habilite GPU se houver páginas digitalizadas. Páginas digitais usam
PyMuPDF; Surya é instalado/carregado somente quando necessário. Uma imagem por vez.

## Etapas e persistência

0. Dependências, módulos e testes antes do livro.
A. Entrada por upload/Drive/caminho e preservação do original.
B. Diagnóstico automático por página.
C. Extração incremental e OCR por necessidade.
D. Limpeza conservadora e decisões explícitas.
E. Texto bruto, limpo, patches, hashes e mapas da origem.
F–G. Segmentação e relatório de validação.
H. Exportação por obra, ZIP e SHA256.

**Drive:** permite retomar depois de o Colab descartar a sessão. Arquivos em
/content são temporários. Execute novamente com o mesmo PDF, slug e metadados:
checkpoints íntegros são reutilizados; falhas podem ser retomadas.

Hífens ambíguos e margens repetidas são sugestões de revisão. Exemplos, conectivos,
negações, pontuação, Unicode e repetições significativas permanecem preservados.

In [ ]:
import os, sys, subprocess, importlib.util, importlib.metadata
from pathlib import Path
import tempfile

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
BASE_PINS = ["PyMuPDF==1.26.7", "Pillow==10.4.0", "tqdm==4.67.1"]
if sys.version_info < (3,10):
    raise RuntimeError("Python >=3.10 é necessário.")
if IN_COLAB:
    loaded={name:name in sys.modules for name in ("fitz","PIL")}
    before={}
    for package in ("PyMuPDF","Pillow"):
        try: before[package]=importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError: before[package]=None
    subprocess.check_call([sys.executable,"-m","pip","install",
                           "--disable-pip-version-check",*BASE_PINS])
    if ((loaded["fitz"] and before["PyMuPDF"]!="1.26.7") or
        (loaded["PIL"] and before["Pillow"]!="10.4.0")):
        raise RuntimeError("Dependência já carregada mudou de versão. Reinicie a sessão "
                           "Colab e execute novamente em ordem.")
else:
    print("Modo local; esta execução não é um teste no Google Colab.")
    for package in ("PyMuPDF","Pillow","tqdm"):
        print(package,importlib.metadata.version(package))

## Configuração da obra

Preencha edição/ano/editora a partir do exemplar. Para outro livro, altere PDF
e metadados, incluindo slug. Pastas associadas a outro PDF/metadados são
preservadas: use novo slug para outra obra, edição ou versão bibliográfica.
Upload recebe exatamente um PDF; drive/path usa o caminho informado.
A retomada pode reutilizar o original já preservado sem novo upload.

In [ ]:
INPUT_MODE = "upload" #@param ["upload", "drive", "path"]
CAMINHO_PDF = "" #@param {type:"string"}
ARMAZENAR_NO_DRIVE = True #@param {type:"boolean"}
RETOMAR_ORIGINAL_SALVO = True #@param {type:"boolean"}
PASTA_DRIVE = "/content/drive/MyDrive/corpus_linguistico" #@param {type:"string"}
SLUG = "coerencia_textual" #@param {type:"string"}
TITULO = "A coerência textual" #@param {type:"string"}
AUTORES = ["Ingedore Villaça Koch","Luiz Carlos Travaglia"]
EDICAO = None
ANO = None
EDITORA = None
ISBN = None
METADADOS={"slug":SLUG,"titulo":TITULO,"autores":AUTORES,
           "edicao":EDICAO,"ano":ANO,"editora":EDITORA,"isbn":ISBN}

## Módulos autocontidos

As três células seguintes contêm todas as fontes de limpeza, segmentação e
processamento. Criam módulos isolados em memória, sem baixar código de um
repositório ou depender de arquivos auxiliares. O hash das fontes participa
das chaves; alterações geram novas execuções e preservam resultados anteriores.

In [ ]:
import hashlib,types
def bootstrap_module(name,source):
    module=types.ModuleType(name)
    module.__file__="<notebook:"+name+">"
    module.EMBEDDED_SOURCE_SHA=hashlib.sha256(source.encode("utf-8")).hexdigest()
    sys.modules[name]=module
    exec(compile(source,module.__file__,"exec"),module.__dict__)
    return module

### Limpeza conservadora, patches e mapas reversíveis

In [ ]:
bootstrap_module('corpus_a_cleaning', r'''
"""Conservative, reversible preparation of scholarly PDF/OCR page records.

This module uses only the Python standard library and is intended to be embedded
verbatim in a notebook.  Original text is immutable.  Offsets count Python Unicode
code points, not UTF-8 bytes.  Cleaning is typography/layout preparation, never
language correction.  All content-affecting operations require explicit opt-in.

Main input: {page_id, page_number, width, height, method, raw_text,
             blocks: [{block_id, raw_text, bbox, line_records, role_hint, ...}]}.
The output preserves source metadata and adds clean_blocks, clean_text, patches,
flags and maps.  Blocks are joined with exactly two newlines in both page views;
even an approved, empty margin block retains its place and its raw provenance.
No Unicode normalization, case conversion, spelling correction, automatic
deletion of repeated headers, or automatic hard-hyphen repair occurs.
"""

from __future__ import annotations

import copy
import hashlib
import json
import math
import re
from collections import defaultdict
from typing import Any, Iterable


CLEANING_VERSION = "1.0.0"
DEFAULT_CLEAN_CONFIG = {
    "normalize_crlf": True,
    "remove_soft_hyphen": False,
    "join_soft_hyphenated_lines": False,
    "approved_dehyphenated_words": [],
    "explicit_replacements": [],
    "join_prose_lines": False,
    "prose_roles": ["prose", "body", "paragraph"],
    "protected_roles": ["example", "linguistic_example", "list", "note",
                        "footnote", "table", "code", "quote", "heading"],
    "line_left_tolerance": 2.0,
    "max_line_gap_font_ratio": 1.2,
    "margin_fraction": 0.08,
    "repeated_margin_min_pages": 3,
    "repeated_margin_min_fraction": 0.50,
    "approved_margin_candidate_ids": [],
    "approved_margin_texts": [],
    "minimum_ocr_confidence": 0.80,
    "fail_on_low_ocr_confidence": False,
    "block_separator": "\n\n",
}


class CleaningIntegrityError(ValueError):
    """Source, edit history, configuration or source mapping is inconsistent."""


class OCRReviewRequired(RuntimeError):
    """Explicit strict policy stopped cleaning a low-confidence OCR record."""


def _hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def _config(config=None) -> dict:
    result = copy.deepcopy(DEFAULT_CLEAN_CONFIG)
    if config:
        unknown = set(config) - set(result)
        if unknown:
            raise CleaningIntegrityError("Unknown cleaning options: " + ", ".join(sorted(unknown)))
        result.update(copy.deepcopy(config))
    json.dumps(result, allow_nan=False)
    for name in ("margin_fraction", "repeated_margin_min_fraction", "minimum_ocr_confidence"):
        value = result[name]
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not 0 <= value <= 1:
            raise CleaningIntegrityError(f"{name} must lie in [0, 1]")
    if not isinstance(result["repeated_margin_min_pages"], int) or result["repeated_margin_min_pages"] < 2:
        raise CleaningIntegrityError("repeated_margin_min_pages must be >= 2")
    if not isinstance(result["block_separator"], str):
        raise CleaningIntegrityError("block_separator must be a string")
    for name in ("normalize_crlf", "remove_soft_hyphen", "join_soft_hyphenated_lines",
                 "join_prose_lines", "fail_on_low_ocr_confidence"):
        if not isinstance(result[name], bool):
            raise CleaningIntegrityError(f"{name} must be a boolean")
    for name in ("line_left_tolerance", "max_line_gap_font_ratio"):
        value = result[name]
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or value < 0:
            raise CleaningIntegrityError(f"{name} must be a finite nonnegative number")
    for item in result["explicit_replacements"]:
        if (not isinstance(item, dict) or not isinstance(item.get("before"), str)
                or not item["before"] or not isinstance(item.get("after"), str)
                or not isinstance(item.get("reason"), str) or not item["reason"].strip()):
            raise CleaningIntegrityError("Each explicit replacement needs nonempty before, after and reason strings")
        if item["before"] == item["after"]:
            raise CleaningIntegrityError("Explicit replacement cannot be a no-op")
    for key in ("approved_dehyphenated_words", "approved_margin_candidate_ids", "approved_margin_texts",
                "prose_roles", "protected_roles"):
        if not isinstance(result[key], list) or any(not isinstance(x, str) for x in result[key]):
            raise CleaningIntegrityError(f"{key} must be a JSON list of strings")
    return result


def _source_text(block: dict) -> str:
    text = block.get("raw_text", block.get("text"))
    if not isinstance(text, str):
        raise CleaningIntegrityError("Every block needs string raw_text (or source text)")
    return text


def _bbox(value):
    if (not isinstance(value, (list, tuple)) or len(value) != 4
            or any(isinstance(x, bool) or not isinstance(x, (int, float)) or not math.isfinite(x) for x in value)):
        return None
    x0, y0, x1, y1 = map(float, value)
    return (x0, y0, x1, y1) if x1 >= x0 and y1 >= y0 else None


def _margin_side(page: dict, block: dict, config: dict):
    height = page.get("height")
    if not isinstance(height, (int, float)) or not math.isfinite(height) or height <= 0:
        return None
    bbox = _bbox(block.get("bbox"))
    if bbox is None:
        return None
    if bbox[1] < 0 or bbox[3] > height:
        return None
    margin = height * config["margin_fraction"]
    if bbox[3] <= margin:
        return "top"
    if bbox[1] >= height - margin:
        return "bottom"
    return None


def detect_repeated_margin_candidates(pages: Iterable[dict], config=None) -> list[dict]:
    """Suggest candidates using repeated text AND marginal geometry.

    Normalized signatures exist only for discovery: source strings are unchanged.
    Pure page numbers are grouped together; other digits remain significant.
    Nothing is approved by this function.  Review candidate occurrences against
    page images before copying an ID into approved_margin_candidate_ids.
    """
    cfg = _config(config)
    pages = list(pages)
    grouped = defaultdict(list)
    seen_page_ids = set()
    for position, page in enumerate(pages):
        page_id = str(page.get("page_id", page.get("page_number", position + 1)))
        if page_id in seen_page_ids:
            raise CleaningIntegrityError("Duplicate page_id in margin detection")
        seen_page_ids.add(page_id)
        for bi, block in enumerate(page.get("blocks", [])):
            side = _margin_side(page, block, cfg)
            if side is None:
                continue
            raw = _source_text(block)
            signature = " ".join(raw.split())
            if not signature:
                continue
            if re.fullmatch(r"[0-9]+", signature):
                signature = "<arabic-page-number>"
            key = (side, signature)
            grouped[key].append({"page_id": page_id,
                                 "page_number": page.get("page_number"),
                                 "block_id": str(block.get("block_id", f"block-{bi + 1}")),
                                 "raw_text": raw, "bbox": list(block["bbox"]),
                                 "raw_sha256": _hash(raw)})
    result = []
    for (side, signature), occurrences in sorted(grouped.items()):
        count = len({x["page_id"] for x in occurrences})
        fraction = count / len(pages) if pages else 0.0
        if count < cfg["repeated_margin_min_pages"] or fraction < cfg["repeated_margin_min_fraction"]:
            continue
        digest = _hash(side + "\0" + signature)[:16]
        result.append({"candidate_id": "margin-" + digest, "side": side,
                       "signature": signature, "page_count": count,
                       "page_fraction": fraction, "occurrences": occurrences,
                       "review_status": "pending", "automatic_removal": False})
    return result


class _EditBuffer:
    """Sequential patches with immutable raw provenance for every output character."""

    def __init__(self, raw: str):
        self.raw = raw
        self.text = raw
        self.origins = [(i, i + 1, False) for i in range(len(raw))]
        self.patches = []

    def raw_span(self, start: int, end: int):
        if not 0 <= start <= end <= len(self.text):
            raise CleaningIntegrityError("Invalid Unicode edit offsets")
        origins = self.origins[start:end]
        if origins:
            return min(x[0] for x in origins), max(x[1] for x in origins)
        boundary = self.origins[start][0] if start < len(self.origins) else len(self.raw)
        return boundary, boundary

    def replace(self, start: int, end: int, expected: str, replacement: str, reason: str):
        if self.text[start:end] != expected:
            raise CleaningIntegrityError("Edit expected text does not match current text")
        if expected == replacement:
            return
        raw_start, raw_end = self.raw_span(start, end)
        patch = {"patch_id": len(self.patches) + 1,
                 "scope": "block", "offset_unit": "unicode_code_point",
                 "current_start": start, "current_end": end,
                 "result_start": start, "result_end": start + len(replacement),
                 "raw_start": raw_start, "raw_end": raw_end,
                 "raw_before": self.raw[raw_start:raw_end],
                 "before": expected, "after": replacement, "reason": reason}
        self.patches.append(patch)
        self.text = self.text[:start] + replacement + self.text[end:]
        self.origins[start:end] = [(raw_start, raw_end, True)] * len(replacement)

    def replace_all(self, before: str, after: str, reason: str):
        matches = [m.start() for m in re.finditer(re.escape(before), self.text)]
        for start in reversed(matches):
            self.replace(start, start + len(before), before, after, reason)

    def span_map(self):
        result = []
        clean_cursor = 0
        raw_cursor = 0
        while clean_cursor < len(self.origins):
            raw_start, raw_end, changed = self.origins[clean_cursor]
            if raw_start > raw_cursor:
                result.append({"raw_start": raw_cursor, "raw_end": raw_start,
                               "clean_start": clean_cursor, "clean_end": clean_cursor,
                               "operation": "delete"})
            run_end = clean_cursor + 1
            if changed:
                while run_end < len(self.origins) and self.origins[run_end] == self.origins[clean_cursor]:
                    run_end += 1
                operation = "replace" if raw_end > raw_start else "insert"
            else:
                while run_end < len(self.origins):
                    nxt = self.origins[run_end]
                    if nxt[2] or nxt[0] != raw_end:
                        break
                    raw_end = nxt[1]
                    run_end += 1
                operation = "equal"
            result.append({"raw_start": raw_start, "raw_end": raw_end,
                           "clean_start": clean_cursor, "clean_end": run_end,
                           "operation": operation})
            raw_cursor = max(raw_cursor, raw_end)
            clean_cursor = run_end
        if raw_cursor < len(self.raw):
            result.append({"raw_start": raw_cursor, "raw_end": len(self.raw),
                           "clean_start": len(self.text), "clean_end": len(self.text),
                           "operation": "delete"})
        return result


_PROTECTED_LINE = re.compile(r"^\s*(?:[\*?#•▪‣]|[-–—]\s|\(?\d+[\).:]\s|\(?[a-zA-Z][\).]\s)")
_HARD_HYPHEN = re.compile(r"([^\W\d_]+)-[ \t]*\n[ \t]*([^\W\d_]+)", re.UNICODE)
_SOFT_HYPHEN_LINE = re.compile(r"([^\W\d_]+)\u00ad[ \t]*\n[ \t]*([^\W\d_]+)", re.UNICODE)


def _protected(block: dict, text: str, cfg: dict) -> bool:
    role = block.get("role_hint", block.get("role", block.get("kind", "")))
    return role in cfg["protected_roles"] or any(_PROTECTED_LINE.match(line) for line in text.splitlines())


def _flag(code, message, raw_start=0, raw_end=0, severity="review"):
    return {"code": code, "message": message, "raw_start": raw_start,
            "raw_end": raw_end, "severity": severity}


def _prose_line_joins(buffer: _EditBuffer, block: dict, cfg: dict, flags: list):
    role = block.get("role_hint", block.get("role", block.get("kind", "")))
    if role not in cfg["prose_roles"] or _protected(block, buffer.text, cfg):
        if "\n" in buffer.text:
            flags.append(_flag("prose_join_skipped_protected_or_untyped", "No prose join: protected or untyped block", 0, len(buffer.raw)))
        return
    records = block.get("line_records", block.get("lines", []))
    raw_lines = buffer.raw.splitlines(keepends=True)
    if (not isinstance(records, list) or len(records) != len(raw_lines)
            or any(not isinstance(x, dict) for x in records)
            or any(rec.get("text") != raw_line.rstrip("\r\n") for rec, raw_line in zip(records, raw_lines))):
        flags.append(_flag("prose_join_missing_exact_line_geometry", "Exact line texts and geometry are required for prose joining", 0, len(buffer.raw)))
        return
    # Origin offsets locate boundaries even after an earlier reviewed change.
    raw_position = 0
    possible = []
    for i, raw_line in enumerate(raw_lines[:-1]):
        boundary_end = raw_position + len(raw_line)
        boundary_start = boundary_end - len(raw_line) + len(raw_line.rstrip("\r\n"))
        raw_position = boundary_end
        left = raw_line.rstrip("\r\n")
        right = raw_lines[i + 1].rstrip("\r\n")
        lb, rb = _bbox(records[i].get("bbox")), _bbox(records[i + 1].get("bbox"))
        fs = records[i].get("font_size")
        rs = records[i + 1].get("font_size")
        if lb is None or rb is None or not isinstance(fs, (int, float)) or fs <= 0:
            continue
        if rs is not None and (not isinstance(rs, (int, float)) or abs(rs - fs) > 0.1):
            continue
        if abs(lb[0] - rb[0]) > cfg["line_left_tolerance"]:
            continue
        if not 0 <= rb[1] - lb[3] <= fs * cfg["max_line_gap_font_ratio"]:
            continue
        if (not left or not right or left != left.rstrip() or right != right.lstrip()
                or left.endswith(("-", "\u00ad", ".", "!", "?", ";", ":"))
                or not left[-1].isalpha() or not right[0].islower()):
            continue
        indices = [j for j, origin in enumerate(buffer.origins)
                   if origin[0] >= boundary_start and origin[1] <= boundary_end]
        if not indices or indices != list(range(indices[0], indices[-1] + 1)):
            continue
        start, end = indices[0], indices[-1] + 1
        if buffer.text[start:end] not in ("\n", "\r\n"):
            continue
        possible.append((start, end))
    for start, end in reversed(possible):
        buffer.replace(start, end, buffer.text[start:end], " ", "reviewed_prose_layout_line_join")


def _candidate_for_block(page_id: str, block_id: str, raw: str, candidates):
    matches = []
    for candidate in candidates:
        for occurrence in candidate.get("occurrences", []):
            if (str(occurrence.get("page_id")) == page_id
                    and str(occurrence.get("block_id")) == block_id
                    and occurrence.get("raw_sha256") == _hash(raw)
                    and occurrence.get("raw_text") == raw):
                matches.append(candidate)
                break
    return matches


def _clean_block(page, block, block_id, cfg, candidates):
    raw = _source_text(block)
    buffer = _EditBuffer(raw)
    flags = []
    for match in re.finditer("\ufffd", raw):
        flags.append(_flag("unicode_replacement_character_preserved", "Replacement character signals possible extraction loss; source image review required", match.start(), match.end()))
    page_id = str(page.get("page_id", page.get("page_number", "page")))
    confidence = block.get("ocr_confidence", page.get("ocr_confidence"))
    is_ocr = "ocr" in str(block.get("method", page.get("method", ""))).lower()
    low_ocr = False
    if is_ocr:
        if confidence is None:
            low_ocr = True
            flags.append(_flag("ocr_confidence_missing", "OCR confidence is absent; content-changing cleanup withheld", 0, len(raw)))
        elif isinstance(confidence, bool) or not isinstance(confidence, (int, float)) or not math.isfinite(confidence) or not 0 <= confidence <= 1:
            raise CleaningIntegrityError("ocr_confidence must lie in [0, 1]; do not infer percentage conversion")
        elif confidence < cfg["minimum_ocr_confidence"]:
            low_ocr = True
            flags.append(_flag("ocr_confidence_below_threshold", "Low-confidence OCR requires image review; content-changing cleanup withheld", 0, len(raw)))
        if low_ocr and cfg["fail_on_low_ocr_confidence"]:
            raise OCRReviewRequired(f"OCR review required for page {page_id}, block {block_id}")
    if cfg["normalize_crlf"]:
        buffer.replace_all("\r\n", "\n", "crlf_to_lf")
    if "\r" in buffer.text:
        flags.append(_flag("bare_carriage_return_preserved", "Bare carriage returns are preserved for review", 0, len(raw)))
    matched_candidates = _candidate_for_block(page_id, block_id, raw, candidates)
    approved_ids = set(cfg["approved_margin_candidate_ids"])
    approved = any(c.get("candidate_id") in approved_ids for c in matched_candidates)
    side = _margin_side(page, block, cfg)
    approved = approved or (raw in cfg["approved_margin_texts"] and side is not None)
    if approved and side is None:
        raise CleaningIntegrityError("Approved margin candidate no longer matches margin geometry")
    if matched_candidates and not approved:
        flags.append(_flag("repeated_margin_pending_review", "Repeated marginal block retained until explicit approval", 0, len(raw)))
    removed = approved and not low_ocr
    if removed:
        buffer.replace(0, len(buffer.text), buffer.text, "", "explicitly_approved_margin_removal")
    elif not low_ocr:
        if cfg["join_soft_hyphenated_lines"] and not _protected(block, buffer.text, cfg):
            for match in reversed(list(_SOFT_HYPHEN_LINE.finditer(buffer.text))):
                buffer.replace(match.start(), match.end(), match.group(0), match.group(1) + match.group(2), "opt_in_soft_hyphen_line_recomposition")
        if cfg["remove_soft_hyphen"]:
            buffer.replace_all("\u00ad", "", "opt_in_soft_hyphen_removal")
        for replacement in cfg["explicit_replacements"]:
            buffer.replace_all(replacement["before"], replacement["after"], "explicit_replacement: " + replacement["reason"])
        matches = list(_HARD_HYPHEN.finditer(buffer.text))
        for match in reversed(matches):
            joined = match.group(1) + match.group(2)
            if joined in cfg["approved_dehyphenated_words"] and not _protected(block, buffer.text, cfg):
                buffer.replace(match.start(), match.end(), match.group(0), joined, "explicitly_approved_hard_hyphen_recomposition")
        if cfg["join_prose_lines"]:
            _prose_line_joins(buffer, block, cfg, flags)
    # Residual ambiguities remain visible, including in protected examples.
    for match in _HARD_HYPHEN.finditer(buffer.text):
        start, end = buffer.raw_span(match.start(), match.end())
        flags.append(_flag("ambiguous_hard_hyphen_preserved", "Hyphen at line break preserved; inspect original image", start, end))
    for match in re.finditer("\u00ad", buffer.text):
        start, end = buffer.raw_span(match.start(), match.end())
        flags.append(_flag("soft_hyphen_preserved", "Soft hyphen preserved; optional documented cleanup available", start, end, "info"))
    if _protected(block, raw, cfg):
        flags.append(_flag("protected_linguistic_or_layout_content", "Examples, markers, lists or declared special layout preserved", 0, len(raw), "info"))
    result = copy.deepcopy(block)
    result.update({"block_id": block_id, "raw_text": raw, "clean_text": buffer.text,
                   "raw_sha256": _hash(raw), "clean_sha256": _hash(buffer.text),
                   "patches": buffer.patches, "span_map": buffer.span_map(),
                   "flags": flags, "review_required": any(x["severity"] == "review" for x in flags),
                   "removed_by_review": removed})
    return result


def clean_page_blocks(page_record: dict, config=None, repeated_margin_candidates=None) -> dict:
    """Return reversible clean page records, never modifying the input.

    Missing/low OCR confidence blocks retain content and are visibly marked for
    review. Set fail_on_low_ocr_confidence=True to enforce a hard stop. Only CRLF
    normalization can occur before that stop policy; it cannot repair OCR errors.
    Re-running this function on its output starts from immutable raw_text, giving
    identical clean text and audit rather than cumulatively editing content.
    """
    cfg = _config(config)
    if not isinstance(page_record, dict) or not isinstance(page_record.get("blocks"), list):
        raise CleaningIntegrityError("Page record requires a blocks list")
    candidates = list(repeated_margin_candidates or [])
    separator = cfg["block_separator"]
    original = separator.join(_source_text(b) for b in page_record["blocks"])
    if "raw_text" in page_record and page_record["raw_text"] != original:
        raise CleaningIntegrityError("Page raw_text must equal the ordered source blocks joined by block_separator")
    blocks = []
    seen = set()
    raw_cursor = clean_cursor = 0
    for i, block in enumerate(page_record["blocks"]):
        block_id = str(block.get("block_id", f"block-{i + 1}"))
        if block_id in seen:
            raise CleaningIntegrityError("Duplicate block_id within page")
        seen.add(block_id)
        clean = _clean_block(page_record, block, block_id, cfg, candidates)
        clean.update({"raw_page_start": raw_cursor, "raw_page_end": raw_cursor + len(clean["raw_text"]),
                      "clean_page_start": clean_cursor, "clean_page_end": clean_cursor + len(clean["clean_text"])})
        for patch in clean["patches"]:
            patch["raw_page_start"] = raw_cursor + patch["raw_start"]
            patch["raw_page_end"] = raw_cursor + patch["raw_end"]
        for flag in clean["flags"]:
            flag["raw_page_start"] = raw_cursor + flag["raw_start"]
            flag["raw_page_end"] = raw_cursor + flag["raw_end"]
        blocks.append(clean)
        raw_cursor += len(clean["raw_text"]) + (len(separator) if i < len(page_record["blocks"]) - 1 else 0)
        clean_cursor += len(clean["clean_text"]) + (len(separator) if i < len(page_record["blocks"]) - 1 else 0)
    result = copy.deepcopy(page_record)
    clean_text = separator.join(b["clean_text"] for b in blocks)
    result.update({"blocks": blocks, "clean_blocks": blocks, "raw_text": original,
                   "clean_text": clean_text, "raw_sha256": _hash(original),
                   "clean_sha256": _hash(clean_text), "cleaning_version": CLEANING_VERSION,
                   "cleaning_config": cfg, "offset_unit": "unicode_code_point",
                   "review_required": any(b["review_required"] for b in blocks),
                   "cleaning_summary": {"block_count": len(blocks),
                                        "patch_count": sum(len(b["patches"]) for b in blocks),
                                        "removed_margin_blocks": sum(b["removed_by_review"] for b in blocks)}})
    validate_clean_record(result)
    return result


def apply_patches(raw_text: str, patches: list[dict]) -> str:
    """Replay sequential patches, checking expected text at every step."""
    text = raw_text
    for number, patch in enumerate(patches, 1):
        if patch.get("patch_id") != number:
            raise CleaningIntegrityError("Patch IDs must be sequential")
        start, end = patch["current_start"], patch["current_end"]
        if not isinstance(start, int) or not isinstance(end, int) or not 0 <= start <= end <= len(text):
            raise CleaningIntegrityError("Patch offsets exceed current text")
        if text[start:end] != patch["before"]:
            raise CleaningIntegrityError("Forward patch validation failed")
        rs, re_ = patch["raw_start"], patch["raw_end"]
        if not 0 <= rs <= re_ <= len(raw_text) or raw_text[rs:re_] != patch["raw_before"]:
            raise CleaningIntegrityError("Raw patch provenance failed")
        if patch["result_start"] != start or patch["result_end"] != start + len(patch["after"]):
            raise CleaningIntegrityError("Result patch offsets are inconsistent")
        text = text[:start] + patch["after"] + text[end:]
    return text


def restore_block(block: dict) -> str:
    """Reverse the audit without reading raw_text, then verify its source hash."""
    text = block["clean_text"]
    for patch in reversed(block["patches"]):
        start, end = patch["result_start"], patch["result_end"]
        if not 0 <= start <= end <= len(text) or text[start:end] != patch["after"]:
            raise CleaningIntegrityError("Reverse patch validation failed")
        text = text[:start] + patch["before"] + text[end:]
    if _hash(text) != block["raw_sha256"]:
        raise CleaningIntegrityError("Restored text does not match original hash")
    return text


def map_clean_span_to_raw(block: dict, clean_start: int, clean_end: int) -> dict:
    """Map a nonempty clean span to a conservative source envelope.

    A replacement may map to more source characters than selected clean text;
    exact=False reports this.  The raw_text slice must remain visible for audit.
    For page offsets use raw_page_start/end. No silent offset coercion occurs.
    """
    if (not isinstance(clean_start, int) or not isinstance(clean_end, int)
            or not 0 <= clean_start < clean_end <= len(block["clean_text"])):
        raise CleaningIntegrityError("Expected a nonempty, in-bounds clean span")
    spans = []
    exact = True
    for item in block["span_map"]:
        lo, hi = max(clean_start, item["clean_start"]), min(clean_end, item["clean_end"])
        if lo >= hi:
            continue
        if item["operation"] == "equal":
            rs = item["raw_start"] + lo - item["clean_start"]
            re_ = rs + hi - lo
        else:
            rs, re_ = item["raw_start"], item["raw_end"]
            exact = False
        spans.append((rs, re_))
    if not spans:
        raise CleaningIntegrityError("Clean span has no provenance")
    rs, re_ = min(s[0] for s in spans), max(s[1] for s in spans)
    source = block["raw_text"][rs:re_]
    exact = exact and source == block["clean_text"][clean_start:clean_end]
    base = block.get("raw_page_start", 0)
    return {"raw_start": rs, "raw_end": re_, "raw_page_start": base + rs,
            "raw_page_end": base + re_, "raw_text": source, "exact": exact}


def map_raw_span_to_clean(block: dict, raw_start: int, raw_end: int) -> dict:
    """Map source evidence to output; removed content returns zero-width spans."""
    if not 0 <= raw_start < raw_end <= len(block["raw_text"]):
        raise CleaningIntegrityError("Expected a nonempty, in-bounds raw span")
    spans = []
    exact = True
    for item in block["span_map"]:
        lo, hi = max(raw_start, item["raw_start"]), min(raw_end, item["raw_end"])
        if lo >= hi:
            continue
        if item["operation"] == "equal":
            cs = item["clean_start"] + lo - item["raw_start"]
            ce = cs + hi - lo
        else:
            cs, ce = item["clean_start"], item["clean_end"]
            exact = False
        spans.append((cs, ce))
    if not spans:
        raise CleaningIntegrityError("Raw span has no provenance")
    cs, ce = min(s[0] for s in spans), max(s[1] for s in spans)
    exact = exact and block["raw_text"][raw_start:raw_end] == block["clean_text"][cs:ce]
    base = block.get("clean_page_start", 0)
    return {"clean_start": cs, "clean_end": ce, "clean_page_start": base + cs,
            "clean_page_end": base + ce, "clean_text": block["clean_text"][cs:ce], "exact": exact}


def validate_clean_record(record: dict) -> bool:
    """Check text hashes, replay, full reversibility, page links and mappings."""
    separator = record["cleaning_config"]["block_separator"]
    blocks = record["clean_blocks"]
    if record["blocks"] != blocks:
        raise CleaningIntegrityError("blocks and clean_blocks aliases diverged")
    for block in blocks:
        raw, clean = block["raw_text"], block["clean_text"]
        if _hash(raw) != block["raw_sha256"] or _hash(clean) != block["clean_sha256"]:
            raise CleaningIntegrityError("Block hash mismatch")
        if apply_patches(raw, block["patches"]) != clean or restore_block(block) != raw:
            raise CleaningIntegrityError("Cleaning is not reversible")
        clean_cursor = 0
        covered_raw = [False] * len(raw)
        for item in block["span_map"]:
            rs, re_, cs, ce = (item[x] for x in ("raw_start", "raw_end", "clean_start", "clean_end"))
            if not 0 <= rs <= re_ <= len(raw) or not 0 <= cs <= ce <= len(clean) or cs != clean_cursor:
                raise CleaningIntegrityError("Invalid or discontinuous span map")
            if item["operation"] == "equal" and raw[rs:re_] != clean[cs:ce]:
                raise CleaningIntegrityError("Equal span map content mismatch")
            for index in range(rs, re_):
                covered_raw[index] = True
            clean_cursor = ce
        if clean_cursor != len(clean) or not all(covered_raw):
            raise CleaningIntegrityError("Span map does not cover source and output")
        rs, re_ = block["raw_page_start"], block["raw_page_end"]
        cs, ce = block["clean_page_start"], block["clean_page_end"]
        if record["raw_text"][rs:re_] != raw or record["clean_text"][cs:ce] != clean:
            raise CleaningIntegrityError("Page/block offsets do not agree")
    for field in ("raw_text", "clean_text"):
        if separator.join(b[field] for b in blocks) != record[field]:
            raise CleaningIntegrityError("Page text and ordered blocks diverged")
        hash_field = "raw_sha256" if field == "raw_text" else "clean_sha256"
        if _hash(record[field]) != record[hash_field]:
            raise CleaningIntegrityError("Page hash mismatch")
    json.dumps(record, ensure_ascii=False, allow_nan=False)
    return True
''')

### Segmentação estrutural com offsets e hierarquia

In [ ]:
bootstrap_module('corpus_a_segmentation', r'''
"""Segmentação estrutural conservadora e rastreável, sem interpretação semântica.

Os offsets são índices de pontos de código Unicode, com fim exclusivo. Cada
unidade aponta para ``page.clean_text`` e para ``clean_blocks[i].clean_text``.
Capítulos/seções são hipóteses editoriais para revisão, nunca fatos garantidos.
O texto-fonte e o mapa reversível de limpeza continuam nos artefatos de página.
Este módulo usa apenas a biblioteca padrão e pode ser embutido em um notebook.
"""

from __future__ import annotations

import hashlib
import json
import math
import re
import statistics
from typing import Any, Iterable


SEGMENTATION_VERSION = "corpus-a-structural-1.0"
OFFSET_UNIT = "unicode_codepoints"
_CLOSERS = "\"'”’»)]}"
_ABBREVIATIONS = re.compile(
    r"\b(?:p\.?\s*ex|e\.?\s*g|i\.?\s*e|et\s+al|etc|"
    r"sr|sra|srta|dr|dra|prof|profa|mr|mrs|ms|"
    r"cf|ex|fig|vol|cap|art|ed|eds|org|orgs|p|pp|n|no|nos|"
    r"trad|trads|aprox|av|tel)\.$", re.IGNORECASE,
)
_TITLES = re.compile(r"\b(?:sr|sra|srta|dr|dra|prof|profa|mr|mrs|ms)\.$", re.I)
_CHAPTER_LABEL = re.compile(r"^(?:cap[ií]tulo|chapter|parte|part|livro|book)\b", re.I)
_SECTION_LABEL = re.compile(r"^(?:se[cç][aã]o|section)\b", re.I)
_SECTION_NUMBER = re.compile(r"^\d+(?:\.\d+)+(?:\.)?\s+\S")
_ROLES = {"heading", "body", "note", "reference", "example", "unknown"}


def _stable_id(kind: str, *parts: Any) -> str:
    encoded = json.dumps([SEGMENTATION_VERSION, kind, *parts],
                         ensure_ascii=False, separators=(",", ":"), sort_keys=True)
    return f"{kind}_{hashlib.sha256(encoded.encode('utf-8')).hexdigest()[:24]}"


def _trim_span(text: str, start: int, end: int) -> tuple[int, int]:
    while start < end and text[start].isspace():
        start += 1
    while end > start and text[end - 1].isspace():
        end -= 1
    return start, end


def _paragraph_spans(text: str) -> list[tuple[int, int]]:
    """Quebras de linha simples permanecem no parágrafo; linhas vazias separam."""
    spans = []
    cursor = 0
    for match in re.finditer(r"(?:\r?\n)[ \t]*(?:\r?\n)+", text):
        start, end = _trim_span(text, cursor, match.start())
        if start < end:
            spans.append((start, end))
        cursor = match.end()
    start, end = _trim_span(text, cursor, len(text))
    if start < end:
        spans.append((start, end))
    return spans


def _sentence_spans(text: str) -> tuple[list[tuple[int, int]], list[dict[str, Any]]]:
    """Heurística conservadora; incertezas são conservadas, não resolvidas por IA.

    Não divide decimais, marcadores de listas, iniciais nem abreviações comuns.
    Reticências e possíveis atribuições após fala mantêm a unidade, com alerta.
    """
    boundaries: list[int] = []
    flags: list[dict[str, Any]] = []
    cursor = 0
    while cursor < len(text):
        if text[cursor] not in ".!?":
            cursor += 1
            continue
        mark_start = cursor
        while cursor < len(text) and text[cursor] in ".!?":
            cursor += 1
        mark_end = cursor
        punctuation = text[mark_start:mark_end]
        prefix = text[:mark_end]
        previous = text[mark_start - 1] if mark_start else ""
        following = text[mark_end] if mark_end < len(text) else ""
        if punctuation == "." and previous.isdigit() and following.isdigit():
            continue
        if punctuation == "." and previous.isalpha() and following.isalpha():
            # O ponto interno em p.ex., U.S.A. e outros tokens não delimita período.
            continue
        if punctuation == ".":
            line = text[text.rfind("\n", 0, mark_start) + 1:mark_end]
            if re.fullmatch(r"\s*(?:\d+(?:\.\d+)*|[IVXLCDM]+)\.", line):
                continue
        close_end = mark_end
        while close_end < len(text) and text[close_end] in _CLOSERS:
            close_end += 1
        next_start = close_end
        while next_start < len(text) and text[next_start].isspace():
            next_start += 1
        next_character = text[next_start] if next_start < len(text) else ""
        if ".." in punctuation:
            flags.append({"start": mark_start, "end": close_end,
                          "reason": "ellipsis_boundary_undetermined"})
            continue
        if punctuation == "." and _ABBREVIATIONS.search(prefix):
            if next_character.isupper() and not _TITLES.search(prefix):
                flags.append({"start": mark_start, "end": close_end,
                              "reason": "abbreviation_may_end_sentence"})
            continue
        if punctuation == "." and re.search(r"\b[A-ZÀ-ÖØ-Þ]\.$", prefix):
            if next_character:
                flags.append({"start": mark_start, "end": close_end,
                              "reason": "initial_or_short_sentence_ambiguous"})
                continue
        if punctuation == "." and re.search(r"\b(?:[A-Za-z]\.){2,}$", prefix):
            continue
        if next_start == len(text):
            boundaries.append(close_end)
            continue
        if next_start == close_end:
            flags.append({"start": mark_start, "end": close_end,
                          "reason": "punctuation_without_boundary_spacing"})
            continue
        tail = text[next_start:]
        if re.match(r"[—–-]\s*[a-zà-öø-ÿ]", tail):
            flags.append({"start": mark_start, "end": close_end,
                          "reason": "possible_dialogue_attribution"})
            continue
        if next_character.islower():
            flags.append({"start": mark_start, "end": close_end,
                          "reason": "punctuation_followed_by_lowercase"})
            continue
        if next_character in ",;:":
            flags.append({"start": mark_start, "end": close_end,
                          "reason": "possible_quotation_continuation"})
            continue
        if next_character.isupper() or next_character.isdigit() or next_character in "—–-\"“«(":
            boundaries.append(close_end)
        else:
            flags.append({"start": mark_start, "end": close_end,
                          "reason": "sentence_boundary_undetermined"})
    result = []
    start = 0
    for boundary in boundaries:
        span = _trim_span(text, start, boundary)
        if span[0] < span[1]:
            result.append(span)
        start = boundary
    span = _trim_span(text, start, len(text))
    if span[0] < span[1]:
        result.append(span)
    return result, flags


def _positive_font(value: Any) -> float | None:
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        value = float(value)
        if math.isfinite(value) and value > 0:
            return value
    return None


def _normal_title(text: str) -> str:
    return " ".join(text.split()).casefold()


def _valid_bbox(value: Any) -> tuple[float, float, float, float] | None:
    if not isinstance(value, (list, tuple)) or len(value) != 4:
        return None
    if not all(isinstance(item, (int, float)) and not isinstance(item, bool)
               and math.isfinite(item) for item in value):
        return None
    left, top, right, bottom = map(float, value)
    return (left, top, right, bottom) if right > left and bottom > top else None


def _geometry_hints(blocks: list[dict[str, Any]]) -> None:
    """Só anota pistas visuais; não altera texto nem afirma função editorial."""
    body_boxes = [_valid_bbox(block.get("bbox")) for block in blocks
                  if block.get("role_hint", "unknown") in {"body", "unknown"}]
    body_boxes = [box for box in body_boxes if box is not None]
    if len(body_boxes) < 2:
        return
    left = statistics.median(box[0] for box in body_boxes)
    right = statistics.median(box[2] for box in body_boxes)
    width = right - left
    if width <= 0:
        return
    for index, block in enumerate(blocks):
        box = _valid_bbox(block.get("bbox"))
        hints = []
        if box is not None:
            centered = abs((box[0] + box[2]) / 2 - (left + right) / 2) <= width * .06
            narrower = box[2] - box[0] <= width * .7
            if centered and narrower:
                hints.append("short_centered_geometry")
            previous = _valid_bbox(blocks[index - 1].get("bbox")) if index else None
            following = _valid_bbox(blocks[index + 1].get("bbox")) if index + 1 < len(blocks) else None
            height = box[3] - box[1]
            if (previous and following and box[1] >= previous[3] and following[1] >= box[3]
                    and box[1] - previous[3] >= height * .8
                    and following[1] - box[3] >= height * .8):
                hints.append("vertically_isolated_geometry")
        block["geometry_heading_hints"] = hints


def _toc_records(toc: Any) -> list[dict[str, Any]]:
    """Aceita o TOC simples do PyMuPDF: [nível, título, página 1-based]."""
    result = []
    for item in toc or []:
        if isinstance(item, dict):
            level, title, page = item.get("level"), item.get("title"), item.get("page")
            if page is None:
                page = item.get("page_number")
        elif isinstance(item, (list, tuple)) and len(item) >= 3:
            level, title, page = item[:3]
        else:
            continue
        if (isinstance(level, int) and not isinstance(level, bool) and level >= 1
                and isinstance(page, int) and not isinstance(page, bool) and page >= 1
                and isinstance(title, str) and title.strip()):
            result.append({"level": level, "title": title, "page_number": page})
    return result


def _heading_candidate(text: str, block: dict[str, Any], page_number: int,
                       body_font: float | None,
                       toc: list[dict[str, Any]]) -> dict[str, Any] | None:
    title = text.strip()
    if not title or len(title) > 300 or len(title.splitlines()) > 4:
        return None
    matches = [entry for entry in toc if entry["page_number"] == page_number
               and _normal_title(entry["title"]) == _normal_title(title)]
    # Tipografia de exemplos, notas e referências não abre nova seção por si só.
    if block.get("role_hint") in {"note", "reference", "example"} and not matches:
        return None
    reasons = []
    suggested_type = None
    level = None
    if matches:
        level = matches[0]["level"]
        suggested_type = "chapter" if level == 1 else "section"
        reasons.append("pdf_toc_exact_title_and_page")
    if _CHAPTER_LABEL.search(title):
        suggested_type = suggested_type or "chapter"
        reasons.append("explicit_chapter_label")
    if _SECTION_LABEL.search(title) or _SECTION_NUMBER.search(title):
        suggested_type = suggested_type or "section"
        reasons.append("explicit_section_label_or_number")
    if block.get("role_hint") == "heading":
        reasons.append("upstream_heading_hint")
    font = _positive_font(block.get("font_size"))
    font_ratio = font / body_font if font and body_font else None
    if font_ratio is not None and font_ratio >= 1.18:
        reasons.append("font_larger_than_body_median")
    geometric_hints = block.get("geometry_heading_hints", [])
    if geometric_hints and len(title) <= 100 and len(title.splitlines()) <= 2:
        # Duas pistas independentes para propor título só por geometria.
        if reasons or len(geometric_hints) >= 2:
            reasons.extend(geometric_hints)
    if not reasons:
        return None
    return {"suggested_type": suggested_type, "toc_level": level,
            "evidence_rules": reasons, "font_size": font,
            "body_font_median": body_font, "font_ratio": font_ratio,
            "bbox": block.get("bbox"), "requires_review": True}


def _align_blocks(page: dict[str, Any]) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Valida offsets fornecidos; só usa busca exata em sequência como alternativa.

    Uma inconsistência substitui todos os blocos por um bloco de página, evitando
    duplicação ou perda de texto. Blocos originais continuam no artefato de origem.
    """
    text = page["clean_text"]
    original_blocks = page.get("clean_blocks") or []
    aligned = []
    issues = []
    cursor = 0
    seen = set()
    invalid = False
    for index, original in enumerate(original_blocks):
        block = dict(original)
        block_id = str(block.get("block_id", f"block-{index + 1}"))
        if block_id in seen:
            invalid = True
            issues.append({"reason": "duplicate_block_id", "block_id": block_id})
        seen.add(block_id)
        clean_text = block.get("clean_text", "")
        if not isinstance(clean_text, str):
            raise TypeError(f"clean_text do bloco {block_id} deve ser str")
        block["block_id"] = block_id
        block["clean_text"] = clean_text
        start, end = block.get("clean_page_start"), block.get("clean_page_end")
        if isinstance(start, int) and isinstance(end, int):
            if not (cursor <= start <= end <= len(text) and text[start:end] == clean_text):
                invalid = True
                issues.append({"reason": "block_page_offsets_do_not_match",
                               "block_id": block_id})
                continue
            basis = "upstream_verified_offsets"
        elif "clean_page_start" in block or "clean_page_end" in block:
            invalid = True
            issues.append({"reason": "invalid_block_page_offsets", "block_id": block_id})
            continue
        else:
            start = text.find(clean_text, cursor) if clean_text else cursor
            if start == -1:
                invalid = True
                issues.append({"reason": "block_text_not_found_in_page", "block_id": block_id})
                continue
            end = start + len(clean_text)
            basis = "sequential_exact_alignment"
        if text[cursor:start].strip():
            # Texto entre blocos é preservado em uma unidade explicitamente não atribuída.
            gap_text = text[cursor:start]
            aligned.append({"block_id": f"page-gap-{cursor}-{start}",
                            "clean_text": gap_text, "clean_page_start": cursor,
                            "clean_page_end": start, "role_hint": "unknown",
                            "alignment_basis": "unassigned_page_text", "source_block": False})
            issues.append({"reason": "page_text_outside_blocks", "start": cursor, "end": start})
        block["clean_page_start"] = start
        block["clean_page_end"] = end
        block["alignment_basis"] = basis
        block["source_block"] = True
        aligned.append(block)
        cursor = end
    if invalid:
        return [{"block_id": "page-fallback", "clean_text": text,
                 "clean_page_start": 0, "clean_page_end": len(text),
                 "role_hint": "unknown", "source_block": False,
                 "alignment_basis": "page_fallback_after_mapping_error"}], issues
    if text[cursor:].strip() or (not original_blocks and text):
        aligned.append({"block_id": f"page-gap-{cursor}-{len(text)}",
                        "clean_text": text[cursor:], "clean_page_start": cursor,
                        "clean_page_end": len(text), "role_hint": "unknown",
                        "alignment_basis": "unassigned_page_text", "source_block": False})
        if original_blocks:
            issues.append({"reason": "page_text_outside_blocks", "start": cursor, "end": len(text)})
    return aligned, issues


def _source_unit(kind: str, text: str, page: dict[str, Any], block: dict[str, Any],
                 start: int, end: int, work_id: str, source_sha256: str) -> dict[str, Any]:
    page_start = block["clean_page_start"] + start
    page_end = block["clean_page_start"] + end
    if text != page["clean_text"][page_start:page_end]:
        raise AssertionError("Offset de unidade não corresponde ao texto de página")
    if text != block["clean_text"][start:end]:
        raise AssertionError("Offset de unidade não corresponde ao texto de bloco")
    text_sha256 = hashlib.sha256(text.encode("utf-8")).hexdigest()
    unit_id = _stable_id(kind, work_id, source_sha256, page["page_id"],
                         page["page_number"], block["block_id"], start, end, text_sha256)
    role = block.get("role_hint", "unknown")
    if role not in _ROLES:
        role = "unknown"
    return {"unit_id": unit_id, "unit_type": kind, "work_id": work_id,
            "source_sha256": source_sha256, "text": text, "text_sha256": text_sha256,
            "page_id": page["page_id"], "page_number": page["page_number"],
            "block_id": block["block_id"], "role_hint": role,
            "span": {"scope": "page.clean_text", "start": page_start,
                     "end": page_end, "offset_unit": OFFSET_UNIT},
            "block_span": {"scope": "block.clean_text", "start": start,
                           "end": end, "offset_unit": OFFSET_UNIT},
            "source_ref": {"work_id": work_id, "source_sha256": source_sha256,
                           "page_id": page["page_id"], "block_id": block["block_id"],
                           "source_block": block["source_block"],
                           "clean_page_start": page_start, "clean_page_end": page_end,
                           "clean_block_start": start, "clean_block_end": end,
                           "raw_recovery": ("source_page_raw_text_and_reversible_block_span_map"
                                            if block["source_block"] else "source_page_raw_text")},
            "alignment_basis": block["alignment_basis"],
            "segmentation_version": SEGMENTATION_VERSION}


def segment_pages(pages: Iterable[dict[str, Any]], work_id: str,
                  source_sha256: str, toc: Any = None) -> dict[str, Any]:
    """Produz capítulos/seções candidatos, parágrafos e períodos, todos rastreáveis.

    ``pages`` contém page_number (1-based), page_id e clean_text; clean_blocks
    pode conter clean_page_start/end, bbox, font_size e role_hint. TOC é opcional.
    Não classifica gênero, não atribui conceitos teóricos e não infere conteúdo.
    Ausência de títulos mantém parent_chapter_id/parent_section_id em None.
    A estrutura candidata só deve ser usada após revisão editorial humana.
    """
    if not isinstance(work_id, str) or not work_id.strip():
        raise ValueError("work_id deve ser uma string não vazia")
    if not isinstance(source_sha256, str) or not source_sha256.strip():
        raise ValueError("source_sha256 deve ser uma string não vazia")
    source_pages = list(pages)
    page_ids = set()
    page_numbers = set()
    for page in source_pages:
        if not isinstance(page.get("clean_text"), str):
            raise TypeError("page.clean_text deve ser str")
        number = page.get("page_number")
        if not isinstance(number, int) or isinstance(number, bool) or number < 1:
            raise ValueError("page_number deve ser inteiro 1-based")
        if not isinstance(page.get("page_id"), str) or not page["page_id"]:
            raise ValueError("page_id estável deve ser string não vazia")
        if page["page_id"] in page_ids or number in page_numbers:
            raise ValueError("page_id e page_number devem ser únicos")
        page_ids.add(page["page_id"])
        page_numbers.add(number)
    source_pages.sort(key=lambda page: page["page_number"])
    result: dict[str, Any] = {
        "schema_version": "1.0", "segmentation_version": SEGMENTATION_VERSION,
        "work_id": work_id, "source_sha256": source_sha256, "offset_unit": OFFSET_UNIT,
        "editorial_structure_status": "candidate_requires_review",
        "chapters": [], "sections": [], "paragraphs": [], "sentences": [],
        "review_items": [], "page_index": [],
    }
    normalized_toc = _toc_records(toc)
    matched_toc_keys = set()
    current_chapter = None
    current_section = None
    section_stack: list[dict[str, Any]] = []

    def review(page: dict[str, Any], reason: str, **details: Any) -> None:
        item = {"review_id": _stable_id("review", work_id, source_sha256,
                                        page["page_id"], reason, details),
                "page_id": page["page_id"], "page_number": page["page_number"],
                "reason": reason, "status": "pending", **details}
        result["review_items"].append(item)

    for page in source_pages:
        blocks, mapping_issues = _align_blocks(page)
        _geometry_hints(blocks)
        for issue in mapping_issues:
            details = dict(issue)
            reason = details.pop("reason")
            review(page, reason, **details)
        fonts = [_positive_font(block.get("font_size")) for block in blocks
                 if block.get("role_hint") != "heading"]
        fonts = [font for font in fonts if font is not None]
        body_font = statistics.median(fonts) if fonts else None
        page_paragraph_ids = []
        page_sentence_ids = []
        for block in blocks:
            paragraph_spans = _paragraph_spans(block["clean_text"])
            for start, end in paragraph_spans:
                paragraph = _source_unit("paragraph", block["clean_text"][start:end],
                                         page, block, start, end, work_id, source_sha256)
                heading = _heading_candidate(paragraph["text"], block,
                                             page["page_number"], body_font, normalized_toc)
                if heading is not None:
                    kind = heading["suggested_type"]
                    if kind is None:
                        kind = "chapter" if current_chapter is None else "section"
                        heading["evidence_rules"].append("provisional_heading_order")
                    structure = _source_unit(kind, paragraph["text"], page, block,
                                             start, end, work_id, source_sha256)
                    structure.update({"title": paragraph["text"], "structure_status": "candidate",
                                      "requires_review": True, "heading_evidence": heading,
                                      "title_paragraph_id": paragraph["unit_id"],
                                      "paragraph_ids": [], "sentence_ids": [],
                                      "start_page_number": page["page_number"],
                                      "end_page_number": page["page_number"]})
                    if kind == "chapter":
                        structure.update({"parent_id": work_id, "parent_chapter_id": None,
                                          "section_ids": []})
                        result["chapters"].append(structure)
                        current_chapter = structure
                        current_section = None
                        section_stack = []
                    else:
                        level = heading["toc_level"] or 2
                        while section_stack and section_stack[-1]["level"] >= level:
                            section_stack.pop()
                        parent_section = section_stack[-1] if section_stack else None
                        structure.update({"level": level,
                                          "parent_id": (parent_section["unit_id"] if parent_section
                                                        else current_chapter["unit_id"] if current_chapter
                                                        else work_id),
                                          "parent_section_id": parent_section["unit_id"] if parent_section else None,
                                          "parent_chapter_id": current_chapter["unit_id"] if current_chapter else None,
                                          "section_ids": []})
                        if parent_section:
                            parent_section["section_ids"].append(structure["unit_id"])
                        if current_chapter:
                            current_chapter["section_ids"].append(structure["unit_id"])
                        result["sections"].append(structure)
                        section_stack.append(structure)
                        current_section = structure
                    for entry in normalized_toc:
                        if (entry["page_number"] == page["page_number"]
                                and _normal_title(entry["title"]) == _normal_title(paragraph["text"])):
                            matched_toc_keys.add((entry["level"], entry["title"], entry["page_number"]))
                    paragraph["heading_candidate_id"] = structure["unit_id"]
                    review(page, "editorial_heading_candidate", unit_id=structure["unit_id"],
                           evidence_rules=heading["evidence_rules"])
                paragraph.update({"parent_id": current_section["unit_id"] if current_section
                                  else current_chapter["unit_id"] if current_chapter else work_id,
                                  "parent_chapter_id": current_chapter["unit_id"] if current_chapter else None,
                                  "parent_section_id": current_section["unit_id"] if current_section else None,
                                  "sentence_ids": [], "structure_status": "text_span_observed"})
                spans, flags = _sentence_spans(paragraph["text"])
                paragraph["split_uncertain"] = bool(flags)
                paragraph["sentence_boundary_flags"] = flags
                if flags:
                    review(page, "uncertain_sentence_boundaries", unit_id=paragraph["unit_id"],
                           flags=flags)
                for sent_start, sent_end in spans:
                    sentence = _source_unit("sentence", paragraph["text"][sent_start:sent_end],
                                            page, block, start + sent_start, start + sent_end,
                                            work_id, source_sha256)
                    sentence_flags = [{**flag, "start": max(0, flag["start"] - sent_start),
                                       "end": min(sent_end - sent_start, flag["end"] - sent_start)}
                                      for flag in flags if flag["start"] < sent_end and flag["end"] > sent_start]
                    sentence.update({"parent_id": paragraph["unit_id"],
                                     "parent_paragraph_id": paragraph["unit_id"],
                                     "parent_chapter_id": paragraph["parent_chapter_id"],
                                     "parent_section_id": paragraph["parent_section_id"],
                                     "paragraph_span": {"start": sent_start, "end": sent_end,
                                                        "offset_unit": OFFSET_UNIT},
                                     "split_uncertain": bool(sentence_flags),
                                     "sentence_boundary_flags": sentence_flags})
                    result["sentences"].append(sentence)
                    paragraph["sentence_ids"].append(sentence["unit_id"])
                    page_sentence_ids.append(sentence["unit_id"])
                result["paragraphs"].append(paragraph)
                page_paragraph_ids.append(paragraph["unit_id"])
                if current_chapter:
                    current_chapter["paragraph_ids"].append(paragraph["unit_id"])
                    current_chapter["sentence_ids"].extend(paragraph["sentence_ids"])
                    current_chapter["end_page_number"] = page["page_number"]
                for section in section_stack:
                    section["paragraph_ids"].append(paragraph["unit_id"])
                    section["sentence_ids"].extend(paragraph["sentence_ids"])
                    section["end_page_number"] = page["page_number"]
        result["page_index"].append({"page_id": page["page_id"], "page_number": page["page_number"],
                                     "clean_text_length": len(page["clean_text"]),
                                     "paragraph_ids": page_paragraph_ids, "sentence_ids": page_sentence_ids,
                                     "mapping_status": "requires_review" if mapping_issues else "verified",
                                     "source_audit": {key: page[key] for key in
                                                      ("extraction_method", "extraction_status", "audit_status",
                                                       "extraction", "audit") if key in page}})
    for entry in normalized_toc:
        if (entry["level"], entry["title"], entry["page_number"]) not in matched_toc_keys:
            result["review_items"].append({
                "review_id": _stable_id("review", work_id, source_sha256, "toc_not_matched", entry),
                "reason": "toc_entry_not_matched_to_extracted_heading", "status": "pending",
                "page_number": entry["page_number"], "toc_entry": entry})
    result["counts"] = {key: len(result[key]) for key in
                        ("chapters", "sections", "paragraphs", "sentences", "review_items")}
    return result


def validate_segmentation(result: dict[str, Any], pages: Iterable[dict[str, Any]]) -> dict[str, Any]:
    """Confere texto/offsets, vínculos, IDs únicos e cobertura não branca de página.

    Não avalia pertinência de títulos. Pode ser usada antes de salvar os JSONL.
    """
    page_map = {page["page_id"]: page for page in pages}
    units = [unit for key in ("chapters", "sections", "paragraphs", "sentences")
             for unit in result[key]]
    unit_map = {unit["unit_id"]: unit for unit in units}
    errors = []
    if len(unit_map) != len(units):
        errors.append("duplicate_unit_id")
    for unit in units:
        if unit.get("text_sha256") != hashlib.sha256(unit["text"].encode("utf-8")).hexdigest():
            errors.append(f"unit_text_hash_mismatch:{unit['unit_id']}")
        page = page_map.get(unit["page_id"])
        if page is None:
            errors.append(f"missing_page:{unit['unit_id']}")
            continue
        span = unit["span"]
        if not (0 <= span["start"] <= span["end"] <= len(page["clean_text"])):
            errors.append(f"invalid_page_span:{unit['unit_id']}")
        elif unit["text"] != page["clean_text"][span["start"]:span["end"]]:
            errors.append(f"page_span_mismatch:{unit['unit_id']}")
        if unit["source_ref"]["source_block"]:
            block = next((block for block in page.get("clean_blocks", [])
                          if str(block.get("block_id")) == unit["block_id"]), None)
            if block is None:
                errors.append(f"missing_source_block:{unit['unit_id']}")
            else:
                block_span = unit["block_span"]
                if unit["text"] != block["clean_text"][block_span["start"]:block_span["end"]]:
                    errors.append(f"block_span_mismatch:{unit['unit_id']}")
        for key in ("parent_chapter_id", "parent_section_id", "parent_paragraph_id"):
            parent_id = unit.get(key)
            if parent_id is not None and parent_id not in unit_map:
                errors.append(f"missing_parent:{unit['unit_id']}:{key}")
        for key in ("paragraph_ids", "sentence_ids", "section_ids"):
            if any(child_id not in unit_map for child_id in unit.get(key, [])):
                errors.append(f"missing_child:{unit['unit_id']}:{key}")
        if unit["unit_type"] == "sentence":
            paragraph = unit_map.get(unit["parent_paragraph_id"])
            if paragraph:
                span = unit["paragraph_span"]
                if unit["text"] != paragraph["text"][span["start"]:span["end"]]:
                    errors.append(f"paragraph_span_mismatch:{unit['unit_id']}")
    for page_id, page in page_map.items():
        spans = sorted((unit["span"]["start"], unit["span"]["end"])
                       for unit in result["paragraphs"] if unit["page_id"] == page_id)
        cursor = 0
        for start, end in spans:
            if start < cursor:
                errors.append(f"paragraph_overlap:{page_id}")
            if page["clean_text"][cursor:start].strip():
                errors.append(f"uncovered_page_text:{page_id}")
            cursor = max(cursor, end)
        if page["clean_text"][cursor:].strip():
            errors.append(f"uncovered_page_text:{page_id}")
    for paragraph in result["paragraphs"]:
        spans = sorted((unit["paragraph_span"]["start"], unit["paragraph_span"]["end"])
                       for unit in result["sentences"]
                       if unit["parent_paragraph_id"] == paragraph["unit_id"])
        cursor = 0
        for start, end in spans:
            if start < cursor or paragraph["text"][cursor:start].strip():
                errors.append(f"sentence_coverage_error:{paragraph['unit_id']}")
            cursor = max(cursor, end)
        if paragraph["text"][cursor:].strip():
            errors.append(f"sentence_coverage_error:{paragraph['unit_id']}")
    return {"valid": not errors, "errors": errors, "checked_units": len(units),
            "checked_pages": len(page_map), "editorial_structure_validated": False}
''')

### Extração, Surya, retomada, auditoria e exportação

In [ ]:
bootstrap_module('corpus_a_core', r'''
"""Incremental, auditable Corpus A pipeline. No embeddings or interpretation."""
from __future__ import annotations

import collections
import gc
import hashlib
import importlib.metadata as metadata
import json
import math
import os
from pathlib import Path
import re
import shutil
import statistics
import subprocess
import sys
import tempfile
import traceback
import zipfile

import fitz
from PIL import Image
from tqdm.auto import tqdm
from corpus_a_cleaning import (CLEANING_VERSION, DEFAULT_CLEAN_CONFIG,
    clean_page_blocks, detect_repeated_margin_candidates, validate_clean_record,
    restore_block, map_clean_span_to_raw)
from corpus_a_segmentation import segment_pages, validate_segmentation, SEGMENTATION_VERSION

PIPELINE_VERSION = "1.0.0"
SURYA_VERSION = "0.17.0"
TRANSFORMERS_VERSION = "4.56.2"
EXTRACTION_DEFAULTS = {
    "ocr_dpi": 180, "max_image_pixels": 16_000_000,
    "min_direct_chars": 40, "max_corrupt_fraction": 0.02,
    "image_coverage_for_ocr": 0.55, "mixed_min_chars": 300,
    "minimum_ocr_confidence": 0.80, "use_surya_layout": False,
    "force_ocr_pages": [], "force_direct_pages": [],
    "save_review_images": False, "allow_cpu_ocr": False,
}

def canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def digest(value):
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()

def sha_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def sha_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def atomic_text(path, text, *, history=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if path.read_bytes().decode("utf-8") == text:
            return
        if history is None:
            raise FileExistsError(f"Preservado arquivo divergente: {path}")
        backup = Path(history) / (path.name + "." + sha_file(path)[:16])
        backup.parent.mkdir(parents=True, exist_ok=True)
        if not backup.exists():
            shutil.copy2(path, backup)
    fd, temporary = tempfile.mkstemp(prefix=".corpus-", dir=path.parent)
    try:
        with os.fdopen(fd, "w", encoding="utf-8", newline="") as f:
            f.write(text)
            f.flush()
            os.fsync(f.fileno())
        os.replace(temporary, path)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)

def atomic_json(path, value, *, history=None):
    atomic_text(path, json.dumps(value, ensure_ascii=False, indent=2,
                                sort_keys=True, allow_nan=False) + "\n", history=history)

def write_jsonl(path, records, *, history=None):
    records = list(records)
    text = "".join(canonical(r) + "\n" for r in records)
    atomic_text(path, text, history=history)
    decoded = [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines()]
    if decoded != records:
        raise ValueError(f"JSONL não passou no round-trip: {path}")

def runtime_info():
    packages = {}
    for name in ("PyMuPDF", "Pillow", "surya-ocr", "transformers", "torch", "tqdm"):
        try:
            packages[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            packages[name] = None
    return {"python": sys.version.split()[0], "packages": packages}

def gpu_info():
    try:
        import torch
        available = bool(torch.cuda.is_available())
        return {"available": available, "torch": torch.__version__,
                "device": torch.cuda.get_device_name(0) if available else None,
                "vram_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2)
                    if available else None}
    except ImportError:
        return {"available": False, "torch": None, "device": None, "vram_gib": None}

def install_surya_if_needed(allow_cpu=False):
    """Colab already supplies CUDA Torch. Do not replace it silently."""
    info = gpu_info()
    if not info["available"] and not allow_cpu:
        raise RuntimeError("OCR requer GPU. No Colab: Ambiente de execução > Alterar tipo > GPU. "
                           "Reexecute a instalação e a extração; páginas concluídas são preservadas.")
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError("PyTorch ausente; selecione um runtime Colab com GPU.") from exc
    major_minor = tuple(int(x) for x in re.match(r"(\d+)\.(\d+)", torch.__version__).groups())
    if not (major_minor >= (2, 7) and major_minor < (3, 0)):
        raise RuntimeError(f"Surya {SURYA_VERSION} requer Torch >=2.7,<3; encontrado {torch.__version__}.")
    expected = {"surya-ocr": SURYA_VERSION, "transformers": TRANSFORMERS_VERSION,
                "Pillow": "10.4.0"}
    missing = []
    for name, version in expected.items():
        try:
            actual = metadata.version(name)
        except metadata.PackageNotFoundError:
            actual = None
        if actual != version:
            missing.append(f"{name}=={version}")
    if missing:
        if "surya" in sys.modules or "transformers" in sys.modules:
            raise RuntimeError("Surya/Transformers já importado com outra versão. Reinicie a sessão "
                               "Colab e execute as células em ordem.")
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                               "--disable-pip-version-check", *missing])
    for name, version in expected.items():
        if metadata.version(name) != version:
            raise RuntimeError(f"Versão incompatível de {name}; reinicie o runtime.")

class SuryaAdapter:
    """Verified Surya 0.17 API; only one page is passed to each predictor."""
    def __init__(self, cache_dir, allow_cpu=False):
        self.cache_dir = str(cache_dir)
        self.allow_cpu = allow_cpu
        self.recognizer = None
        self.detector = None
        self.provenance = {}

    def _prepare(self):
        install_surya_if_needed(self.allow_cpu)
        os.environ["MODEL_CACHE_DIR"] = self.cache_dir
        os.environ["TORCH_DEVICE"] = "cuda" if gpu_info()["available"] else "cpu"
        os.environ["DETECTOR_BATCH_SIZE"] = "1"
        os.environ["RECOGNITION_BATCH_SIZE"] = "1"
        os.environ["LAYOUT_BATCH_SIZE"] = "1"
        os.environ["DISABLE_TQDM"] = "true"
        return os.environ["TORCH_DEVICE"]

    def release(self):
        self.recognizer = None
        self.detector = None
        gc.collect()
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except ImportError:
            pass

    def recognize(self, image):
        device = self._prepare()
        if self.recognizer is None:
            from surya.foundation import FoundationPredictor
            from surya.recognition import RecognitionPredictor
            from surya.detection import DetectionPredictor
            from surya.settings import settings
            expected={"FOUNDATION_MODEL_CHECKPOINT":"s3://text_recognition/2025_09_23",
                      "DETECTOR_MODEL_CHECKPOINT":"s3://text_detection/2025_05_07"}
            for name,value in expected.items():
                if getattr(settings,name)!=value:
                    raise RuntimeError(f"Checkpoint {name} divergente da versão examinada; reinicie sem override.")
            foundation = FoundationPredictor(device=device, attention_implementation="sdpa")
            self.recognizer = RecognitionPredictor(foundation)
            self.detector = DetectionPredictor(device=device)
            self.provenance.update({"surya_version":metadata.version("surya-ocr"),
                "transformers_version":metadata.version("transformers"),"device":device,
                "recognition_checkpoint":settings.FOUNDATION_MODEL_CHECKPOINT,
                "detection_checkpoint":settings.DETECTOR_MODEL_CHECKPOINT})
        result = self.recognizer([image], det_predictor=self.detector,
            detection_batch_size=1, recognition_batch_size=1, math_mode=False,
            sort_lines=False, return_words=False, drop_repeated_text=False)[0]
        return [{"text": str(line.text), "bbox_image": [float(v) for v in line.bbox],
                 "polygon_image": [[float(x), float(y)] for x, y in line.polygon],
                 "confidence": float(line.confidence)} for line in result.text_lines]

    def layout(self, image):
        self.release()  # Layout uses another Foundation; do not retain both in VRAM.
        device = self._prepare()
        from surya.foundation import FoundationPredictor
        from surya.layout import LayoutPredictor
        from surya.settings import settings
        if settings.LAYOUT_MODEL_CHECKPOINT!="s3://layout/2025_09_23":
            raise RuntimeError("Checkpoint de layout divergente da versão examinada.")
        self.provenance["layout_checkpoint"]=settings.LAYOUT_MODEL_CHECKPOINT
        foundation = FoundationPredictor(checkpoint=settings.LAYOUT_MODEL_CHECKPOINT,
                                         device=device, attention_implementation="sdpa")
        predictor = LayoutPredictor(foundation)
        try:
            result = predictor([image], batch_size=1)[0]
            return [{"label": b.label, "position": int(b.position),
                     "bbox_image": [float(v) for v in b.bbox],
                     "confidence": float(b.confidence) if b.confidence is not None else None}
                    for b in result.bboxes]
        finally:
            del predictor, foundation
            gc.collect()
            try:
                import torch
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except ImportError:
                pass

def transformed_bbox(page, bbox):
    return [round(float(v), 3) for v in fitz.Rect(bbox) * page.rotation_matrix]

def multicolumn_possible(items, width):
    left = [x for x in items if x["bbox"][0] < width * .4 and
            x["bbox"][2] - x["bbox"][0] < width * .58]
    right = [x for x in items if x["bbox"][0] > width * .44]
    if len(left) < 2 or len(right) < 2:
        return False
    return max(min(x["bbox"][1] for x in left), min(x["bbox"][1] for x in right)) < min(
        max(x["bbox"][3] for x in left), max(x["bbox"][3] for x in right))

def role_hint(text, font_size, median_font, bbox, height):
    if re.match(r"\s*(?:\(?\d+[a-z]?\)|[*?#]|[a-z]\))\s*", text):
        return "example"
    if re.match(r"\s*(?:CAP[IÍ]TULO|SE[CÇ][AÃ]O|REFER[EÊ]NCIAS|BIBLIOGRAFIA)\b", text, re.I):
        return "heading"
    if font_size and median_font and font_size > median_font * 1.25 and len(text) < 250:
        return "heading"
    if font_size and median_font and font_size < median_font * .86 and bbox[1] > height * .60:
        return "note"
    return "body"

def digital_blocks(page):
    data = page.get_text("dict", flags=fitz.TEXTFLAGS_DICT & ~fitz.TEXT_PRESERVE_IMAGES,
                         sort=False)
    sizes = [s.get("size", 0) for b in data["blocks"] if b.get("type") == 0
             for line in b.get("lines", []) for s in line.get("spans", []) if s.get("text", "").strip()]
    median_font = statistics.median(sizes) if sizes else 0
    blocks = []
    for source_index, b in enumerate(data["blocks"]):
        if b.get("type") != 0:
            continue
        lines = []
        for line in b.get("lines", []):
            text = "".join(s.get("text", "") for s in line.get("spans", []))
            lines.append({"text": text, "bbox": transformed_bbox(page, line["bbox"]),
                          "font_size": statistics.median([s.get("size", 0) for s in line["spans"]])
                               if line.get("spans") else 0,
                          "spans": [{"text": s.get("text", ""), "font": s.get("font"),
                                     "font_size": s.get("size"), "flags": s.get("flags"),
                                     "bbox_pdf_unrotated": list(s.get("bbox", []))}
                                    for s in line.get("spans", [])]})
        raw = "\n".join(x["text"] for x in lines)
        if not raw.strip():
            continue
        bbox = transformed_bbox(page, b["bbox"])
        size = statistics.median([x["font_size"] for x in lines]) if lines else 0
        blocks.append({"raw_text": raw, "bbox": bbox,
                       "bbox_pdf_unrotated": list(b["bbox"]), "font_size": size,
                       "line_records": lines, "source_index": source_index,
                       "role_hint": role_hint(raw, size, median_font, bbox, page.rect.height),
                       "role_status": "candidate", "source": "pymupdf"})
    return sorted(blocks, key=lambda b: (round(b["bbox"][1], 1), b["bbox"][0], b["source_index"]))

def inspect_page(page, blocks, config):
    text = "\n\n".join(b["raw_text"] for b in blocks)
    n = len(text.strip())
    bad = sum(c == "\ufffd" or (ord(c) < 32 and c not in "\n\r\t") or 0x7f <= ord(c) <= 0x9f for c in text)
    image_boxes = [transformed_bbox(page, im["bbox"]) for im in page.get_image_info()]
    coverage = min(1.0, sum(max(0, b[2]-b[0])*max(0, b[3]-b[1]) for b in image_boxes)
                   / max(1, page.rect.width * page.rect.height))
    corrupt = bad / max(1, n)
    vector_objects = len(page.get_drawings())
    if n == 0 and not image_boxes and not vector_objects:
        kind, needs_ocr = "sem_texto", False
    elif n == 0:
        kind, needs_ocr = "digitalizado", True
    elif coverage >= config["image_coverage_for_ocr"]:
        kind = "misto"
        needs_ocr = n < config["mixed_min_chars"] or corrupt > config["max_corrupt_fraction"]
    else:
        kind = "digital"
        needs_ocr = corrupt > config["max_corrupt_fraction"]
    flags = []
    if n < config["min_direct_chars"]:
        flags.append({"code": "few_native_characters", "message": "Página sem/muito pouco texto digital; conferir."})
    if bad:
        flags.append({"code": "suspicious_native_characters", "count": bad})
    if image_boxes:
        flags.append({"code": "image_regions", "message": "Imagens podem conter conteúdo não textual ou exemplos; revisar.",
                      "image_count": len(image_boxes), "coverage_estimate": coverage})
    if n == 0 and vector_objects:
        flags.append({"code": "vector_content_without_text", "count":vector_objects,
                      "message":"Conteúdo vetorial sem texto: OCR necessário e revisão visual."})
    native_lines=[line for b in blocks for line in b.get("line_records",[])]
    if multicolumn_possible(native_lines or blocks, page.rect.width):
        flags.append({"code": "possible_multiple_columns", "message": "Ordem geométrica exige conferência."})
    return {"page_kind": kind, "needs_ocr": needs_ocr, "native_chars": n,
            "corrupt_fraction": corrupt, "image_coverage_estimate": coverage,
            "image_boxes": image_boxes, "vector_objects":vector_objects,"flags": flags}

def render_page(page, config):
    zoom = config["ocr_dpi"] / 72
    pixels = page.rect.width * page.rect.height * zoom**2
    if pixels > config["max_image_pixels"]:
        zoom *= math.sqrt(config["max_image_pixels"] / pixels)
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), colorspace=fitz.csRGB, alpha=False)
    image = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    del pix
    return image

def bbox_coverage(small, large):
    x0, y0 = max(small[0], large[0]), max(small[1], large[1])
    x1, y1 = min(small[2], large[2]), min(small[3], large[3])
    return max(0, x1-x0) * max(0, y1-y0) / max(1, (small[2]-small[0])*(small[3]-small[1]))

LAYOUT_ROLES = {"SectionHeader": "heading", "Footnote": "note", "Table": "table",
    "Equation": "example", "Code": "code", "ListItem": "list", "Caption": "note",
    "PageHeader": "header_candidate", "PageFooter": "footer_candidate", "Text": "body"}

def blocks_from_ocr(lines, page, image, layout_boxes=None):
    """Conserve every recognized line; grouping/order are explicitly candidates."""
    sx, sy = page.rect.width / image.width, page.rect.height / image.height
    records = []
    for i, line in enumerate(lines):
        record = dict(line)
        record["source_index"] = i
        record["bbox"] = [line["bbox_image"][0]*sx, line["bbox_image"][1]*sy,
                          line["bbox_image"][2]*sx, line["bbox_image"][3]*sy]
        record["font_size"] = max(1, record["bbox"][3]-record["bbox"][1])
        records.append(record)
    records.sort(key=lambda r: (round(r["bbox"][1], 1), r["bbox"][0]))
    groups, unassigned = [], []
    if layout_boxes:
        buckets = collections.defaultdict(list)
        for line in records:
            overlaps = [bbox_coverage(line["bbox_image"], x["bbox_image"]) for x in layout_boxes]
            winner = max(range(len(overlaps)), key=overlaps.__getitem__)
            if overlaps[winner] >= .50:
                buckets[winner].append(line)
            else:
                unassigned.append(line)
        for i in sorted(buckets, key=lambda n: layout_boxes[n]["position"]):
            box = layout_boxes[i]
            groups.append((buckets[i], LAYOUT_ROLES.get(box["label"], "unknown"), box))
    else:
        median_height = statistics.median([r["font_size"] for r in records]) if records else 12
        for line in records:
            hint = role_hint(line["text"], line["font_size"], median_height,
                             line["bbox"], page.rect.height)
            if groups and hint == "body" and groups[-1][1] == "body":
                previous = groups[-1][0][-1]
                if (0 <= line["bbox"][1]-previous["bbox"][3] <= median_height*.7 and
                    abs(line["bbox"][0]-previous["bbox"][0]) <= median_height*.7):
                    groups[-1][0].append(line)
                    continue
            groups.append(([line], hint, None))
    for line in unassigned:
        groups.append(([line], "unknown", None))
    blocks = []
    for group, hint, box in groups:
        group.sort(key=lambda r: (round(r["bbox"][1], 1), r["bbox"][0]))
        bbox = [min(r["bbox"][0] for r in group), min(r["bbox"][1] for r in group),
                max(r["bbox"][2] for r in group), max(r["bbox"][3] for r in group)]
        blocks.append({"raw_text": "\n".join(r["text"] for r in group), "bbox": bbox,
                       "line_records": group, "role_hint": hint, "role_status": "candidate",
                       "confidence": statistics.mean(r["confidence"] for r in group),
                       "ocr_confidence": min(r["confidence"] for r in group),
                       "source": "surya", "layout": box,
                       "unassigned_layout": bool(layout_boxes and box is None)})
    if sum(len(b["line_records"]) for b in blocks) != len(lines):
        raise ValueError("Perda/duplicação de linhas na associação ao layout.")
    return blocks

def layout_order_digital(blocks, layout_boxes, page, image):
    for b in blocks:
        ibox = [b["bbox"][0]*image.width/page.rect.width,
                b["bbox"][1]*image.height/page.rect.height,
                b["bbox"][2]*image.width/page.rect.width,
                b["bbox"][3]*image.height/page.rect.height]
        overlaps = [bbox_coverage(ibox, x["bbox_image"]) for x in layout_boxes]
        if overlaps and max(overlaps) >= .5:
            item = layout_boxes[max(range(len(overlaps)), key=overlaps.__getitem__)]
            b["layout"] = item
            b["layout_position"] = item["position"]
            b["role_hint"] = LAYOUT_ROLES.get(item["label"], b["role_hint"])
        else:
            b["layout_position"] = 1_000_000 + b["source_index"]
            b["unassigned_layout"] = True
    return sorted(blocks, key=lambda b: (b["layout_position"], b["bbox"][1], b["bbox"][0]))

def finalize_raw_record(record):
    for i, b in enumerate(record.get("blocks", [])):
        b["order"] = i
        b["block_id"] = f'{record["page_id"]}:b{i:04d}'
    record["raw_text"] = "\n\n".join(b["raw_text"] for b in record.get("blocks", []))
    record["raw_sha256"] = sha_text(record["raw_text"])
    record.pop("record_sha256", None)
    record["record_sha256"] = digest(record)
    return record

def extract_page(page, page_id, config, adapter, extract_key):
    blocks = digital_blocks(page)
    inspection = inspect_page(page, blocks, config)
    number = page.number + 1
    needs_ocr = inspection["needs_ocr"]
    if number in config["force_ocr_pages"]:
        needs_ocr = True
    if number in config["force_direct_pages"]:
        needs_ocr = False
    if number in config["force_ocr_pages"] and number in config["force_direct_pages"]:
        raise ValueError("Página presente em force_ocr_pages e force_direct_pages.")
    record = {"schema_version": "corpus_a.page.v1", "extract_key": extract_key,
              "page_number": number, "page_id": page_id, "width": page.rect.width,
              "height": page.rect.height, "rotation": page.rotation,
              "coordinates": "PDF display coordinates; page number 1-based",
              "inspection": inspection, "blocks": blocks, "status": "ok",
              "method": "digital", "flags": list(inspection["flags"]),
              "reading_order": "geometry_yx_candidate", "errors": []}
    image = None
    try:
        if needs_ocr:
            record["native_alternative"] = {"blocks": blocks,
                                            "text": "\n\n".join(b["raw_text"] for b in blocks)}
            image = render_page(page, config)
            layout_boxes = adapter.layout(image) if config["use_surya_layout"] else None
            lines = adapter.recognize(image)
            record["blocks"] = blocks_from_ocr(lines, page, image, layout_boxes)
            record["method"] = "ocr"
            record["ocr_lines"] = lines
            record["render_size"] = [image.width, image.height]
            record["reading_order"] = "surya_layout_candidate" if layout_boxes else "geometry_yx_candidate"
            record["flags"].append({"code": "ocr_requires_review", "message": "OCR não comprova fidelidade editorial."})
            low = [i for i, line in enumerate(lines) if line["confidence"] < config["minimum_ocr_confidence"]]
            if low:
                record["flags"].append({"code": "low_ocr_confidence", "line_indices": low})
            if not layout_boxes and multicolumn_possible(record["blocks"], page.rect.width):
                record["flags"].append({"code": "ocr_multicolumn_without_layout"})
        elif inspection["page_kind"] == "sem_texto":
            record["method"] = "blank"
        elif config["use_surya_layout"] and any(f["code"]=="possible_multiple_columns" for f in inspection["flags"]):
            image = render_page(page, config)
            layout_boxes = adapter.layout(image)
            record["blocks"] = layout_order_digital(blocks, layout_boxes, page, image)
            record["reading_order"] = "surya_layout_candidate"
        if any(b.get("unassigned_layout") for b in record["blocks"]):
            record["flags"].append({"code": "layout_unassigned_content"})
    except Exception as exc:
        # Preserve the native alternative; never label failed OCR as successful.
        record["status"] = "error"
        record["method"] = "ocr_error" if needs_ocr else "layout_error"
        record["blocks"] = blocks
        record["errors"] = [{"type": type(exc).__name__, "message": str(exc)}]
        record["flags"].append({"code": "extraction_incomplete"})
    finally:
        if image is not None:
            image.close()
        gc.collect()
    record["extraction_runtime"]=runtime_info()
    record["ocr_adapter"]=type(adapter).__name__ if needs_ocr or config["use_surya_layout"] else None
    record["ocr_provenance"]=dict(getattr(adapter,"provenance",{}))
    return finalize_raw_record(record)

class CorpusPipeline:
    def __init__(self, pdf_path, output_base, bibliography, *, extraction=None,
                 cleaning=None, adapter=None, cache_dir=None):
        self.source = Path(pdf_path).expanduser().resolve()
        if not self.source.is_file():
            raise FileNotFoundError(f"PDF ausente: {self.source}")
        self.bibliography = dict(bibliography)
        self.slug = self.bibliography["slug"]
        if not re.fullmatch(r"[a-z0-9][a-z0-9_-]*", self.slug):
            raise ValueError("slug: use letras minúsculas ASCII, números, _ ou -.")
        self.source_sha = sha_file(self.source)
        self.work_id = f"obra-{self.source_sha[:24]}"
        self.root = Path(output_base) / "obras" / self.slug
        self.extraction = dict(EXTRACTION_DEFAULTS)
        if extraction:
            if set(extraction)-set(self.extraction):
                raise ValueError("Opções de extração desconhecidas.")
            self.extraction.update(extraction)
        self.cleaning = dict(DEFAULT_CLEAN_CONFIG)
        if cleaning:
            self.cleaning.update(cleaning)
        self.adapter = adapter or SuryaAdapter(cache_dir or Path(output_base)/"cache_surya",
                                               self.extraction["allow_cpu_ocr"])
        self._initialize()
        self.extract_key = digest({"pipeline": PIPELINE_VERSION, "source": self.source_sha,
                                  "code_sha256":globals().get("EMBEDDED_SOURCE_SHA",PIPELINE_VERSION),
                                  "config": self.extraction, "pymupdf": metadata.version("PyMuPDF"),
                                  "surya_api": SURYA_VERSION})[:24]
        self.clean_key = digest({"extract": self.extract_key, "version": CLEANING_VERSION,
                                "code_sha256":getattr(sys.modules["corpus_a_cleaning"],"EMBEDDED_SOURCE_SHA",CLEANING_VERSION),
                                "config": self.cleaning})[:24]
        self.run_id = digest({"extract": self.extract_key, "clean": self.clean_key,
                              "segmentation": SEGMENTATION_VERSION,
                              "segment_code_sha256":getattr(sys.modules["corpus_a_segmentation"],"EMBEDDED_SOURCE_SHA",SEGMENTATION_VERSION)})[:24]
        self.raw_dir = self.root/"extraido"/"execucoes"/self.extract_key/"paginas"
        self.clean_dir = self.root/"limpo"/"execucoes"/self.clean_key/"paginas"
        self.segment_dir = self.root/"segmentado"/"execucoes"/self.run_id
        self.audit_dir = self.root/"auditoria"/"execucoes"/self.run_id
        self.report_dir = self.root/"relatorios"/self.run_id
        for p in (self.raw_dir, self.clean_dir, self.segment_dir, self.audit_dir, self.report_dir):
            p.mkdir(parents=True, exist_ok=True)

    def _initialize(self):
        for folder in ("original", "extraido", "limpo", "segmentado", "auditoria", "relatorios"):
            (self.root/folder).mkdir(parents=True, exist_ok=True)
        manifest_path = self.root/"manifesto.json"
        if manifest_path.exists():
            manifest = load_json(manifest_path)
            if manifest["source_sha256"] != self.source_sha:
                raise ValueError("Esta pasta já pertence a outro PDF. Use novo slug; nenhum original foi substituído.")
            if manifest["bibliography"] != self.bibliography:
                raise ValueError("Metadados diferentes dos registrados; preserve a pasta e use novo slug/versionamento.")
        original = self.root/"original"/(self.source_sha+".pdf")
        if original.exists() and sha_file(original) != self.source_sha:
            raise ValueError("Cópia original divergente: interrompido para preservar evidência.")
        if not original.exists():
            temporary = original.with_suffix(".pdf.part")
            shutil.copyfile(self.source, temporary)
            if sha_file(temporary) != self.source_sha:
                raise ValueError("Falha de integridade ao copiar PDF.")
            os.replace(temporary, original)
        self.original = original
        with fitz.open(original) as doc:
            if doc.needs_pass:
                raise ValueError("PDF protegido por senha; forneça cópia acessível mantendo o original.")
            self.page_count = len(doc)
            self.pdf_metadata = dict(doc.metadata or {})
            self.toc = doc.get_toc(simple=True)
        if self.page_count < 1:
            raise ValueError("PDF sem páginas.")
        if not manifest_path.exists():
            atomic_json(manifest_path, {"schema_version": "corpus_a.manifest.v1",
                "work_id": self.work_id, "source_sha256": self.source_sha,
                "bibliography": self.bibliography, "page_count": self.page_count,
                "original": str(original.relative_to(self.root)),
                "pdf_metadata": self.pdf_metadata, "toc": self.toc, "runs": {}})

    def raw_path(self, number):
        return self.raw_dir/f"pagina_{number:05d}.json"

    def clean_path(self, number):
        return self.clean_dir/f"pagina_{number:05d}.json"

    def read_raw(self):
        result=[]
        for n in range(1,self.page_count+1):
            path=self.raw_path(n)
            if not path.exists():
                continue
            record=load_json(path)
            expected=digest({k:v for k,v in record.items() if k!="record_sha256"})
            if (record.get("extract_key")!=self.extract_key or record.get("record_sha256")!=expected or
                record.get("raw_sha256")!=sha_text(record.get("raw_text","")) or
                record.get("page_number")!=n or record.get("page_id")!=f"{self.work_id}:p{n:05d}"):
                raise ValueError(f"Checkpoint divergente: {path}")
            result.append(record)
        return result

    def extract(self, page_numbers=None, *, retry_errors=True):
        numbers = list(page_numbers) if page_numbers is not None else list(range(1, self.page_count+1))
        if len(set(numbers)) != len(numbers) or any(not isinstance(n, int) or not 1 <= n <= self.page_count for n in numbers):
            raise ValueError("Números de página inválidos; usam base 1.")
        summary = collections.Counter()
        with fitz.open(self.original) as doc:
            for number in tqdm(numbers, desc="Extração por página"):
                path = self.raw_path(number)
                if path.exists():
                    previous = load_json(path)
                    if (previous.get("extract_key") != self.extract_key or
                        previous.get("raw_sha256") != sha_text(previous.get("raw_text", "")) or
                        previous.get("record_sha256") != digest({k:v for k,v in previous.items() if k != "record_sha256"})):
                        raise ValueError(f"Checkpoint divergente na página {number}.")
                    if previous["status"] == "ok" or not retry_errors:
                        summary["reused"] += 1
                        continue
                try:
                    record = extract_page(doc[number-1], f"{self.work_id}:p{number:05d}",
                                          self.extraction, self.adapter, self.extract_key)
                except Exception as exc:
                    page = doc[number-1]
                    record = finalize_raw_record({"schema_version": "corpus_a.page.v1",
                        "extract_key": self.extract_key, "page_number": number,
                        "page_id": f"{self.work_id}:p{number:05d}", "width": page.rect.width,
                        "height": page.rect.height, "blocks": [], "status": "error",
                        "method": "page_error", "inspection": {}, "flags": [{"code":"extraction_incomplete"}],
                        "errors": [{"type":type(exc).__name__, "message":str(exc)}]})
                atomic_json(path, record, history=self.audit_dir/"tentativas")
                if self.extraction["save_review_images"] and record.get("flags"):
                    preview=render_page(doc[number-1],{**self.extraction,"ocr_dpi":110})
                    try:
                        target=self.audit_dir/"imagens_revisao"/f"pagina_{number:05d}.png"
                        target.parent.mkdir(parents=True,exist_ok=True)
                        preview.save(target)
                    finally:
                        preview.close()
                summary[record["status"]] += 1
        if hasattr(self.adapter, "release"):
            self.adapter.release()
        atomic_json(self.audit_dir/"configuracao.json", {
            "pipeline_version": PIPELINE_VERSION, "run_id": self.run_id,
            "extract_key": self.extract_key, "clean_key": self.clean_key,
            "extraction": self.extraction, "cleaning": self.cleaning,
            "runtime": runtime_info(), "gpu": gpu_info(), "bibliography": self.bibliography},
            history=self.audit_dir/"historico")
        return dict(summary)

    def clean(self):
        raw_pages = self.read_raw()
        candidates = detect_repeated_margin_candidates(raw_pages, self.cleaning)
        candidates_sha=digest(candidates)
        atomic_json(self.audit_dir/"candidatos_margens.json", candidates,
                    history=self.audit_dir/"historico")
        pages = []
        for raw in tqdm(raw_pages, desc="Limpeza auditável"):
            path = self.clean_path(raw["page_number"])
            if path.exists():
                saved = load_json(path)
                validate_clean_record(saved)
                if saved.get("clean_record_sha256") != digest({k:v for k,v in saved.items() if k != "clean_record_sha256"}):
                    raise ValueError(f"Checkpoint limpo divergente: {path}")
                if (saved.get("source_record_sha256") == digest(raw) and saved.get("clean_key") == self.clean_key and
                    saved.get("margin_candidates_sha256")==candidates_sha):
                    pages.append(saved)
                    continue
            try:
                record = clean_page_blocks(raw, self.cleaning, candidates)
                record.update({"clean_key":self.clean_key, "source_record_sha256":digest(raw),
                               "margin_candidates_sha256":candidates_sha,
                               "work_id":self.work_id, "source_sha256":self.source_sha})
                record["clean_record_sha256"] = digest(record)
                validate_clean_record(record)
                atomic_json(path, record, history=self.audit_dir/"historico_limpeza")
                pages.append(record)
            except Exception as exc:
                # Keep this as a cleaning failure, rather than producing unverifiable text.
                atomic_json(self.audit_dir/f"erro_limpeza_{raw['page_number']:05d}.json",
                    {"page_number":raw["page_number"], "type":type(exc).__name__, "message":str(exc)},
                    history=self.audit_dir/"historico")
        return pages, candidates

    def export(self, pages):
        raw_pages = self.read_raw()
        current_raw={p["page_number"]:p for p in raw_pages}
        for p in pages:
            if (p.get("source_record_sha256")!=digest(current_raw[p["page_number"]]) or
                p.get("clean_record_sha256")!=digest({k:v for k,v in p.items() if k!="clean_record_sha256"})):
                raise ValueError("Extração/limpeza divergentes. Reexecute a limpeza após retomar a extração.")
        units = segment_pages(pages, self.work_id, self.source_sha, self.toc)
        history = self.audit_dir/"historico_exportacao"
        for label, records in (("paginas", pages), ("capitulos", units["chapters"]),
                               ("secoes", units["sections"]), ("paragrafos", units["paragraphs"]),
                               ("periodos", units["sentences"])):
            enriched = [{**r, "obra":self.bibliography, "run_id":self.run_id} for r in records]
            write_jsonl(self.segment_dir/(label+".jsonl"), enriched, history=history)
        write_jsonl(self.root/"extraido"/"execucoes"/self.extract_key/"paginas.jsonl", raw_pages, history=history)
        write_jsonl(self.audit_dir/"revisao_segmentacao.jsonl", units["review_items"], history=history)
        changes, suspicious = [], []
        for p in pages:
            for b in p.get("clean_blocks", []):
                for patch in b.get("patches", []):
                    changes.append({"page_number":p["page_number"], "page_id":p["page_id"],
                                    "block_id":b["block_id"], **patch})
                for flag in b.get("flags", []):
                    suspicious.append({"page_number":p["page_number"], "block_id":b["block_id"], **flag})
            raw = next(r for r in raw_pages if r["page_number"] == p["page_number"])
            for text, folder in ((raw["raw_text"], self.raw_dir.parent), (p["clean_text"], self.clean_dir.parent)):
                atomic_text(folder/f"pagina_{p['page_number']:05d}.txt", text, history=history)
        write_jsonl(self.audit_dir/"alteracoes.jsonl", changes, history=history)
        write_jsonl(self.audit_dir/"trechos_suspeitos.jsonl", suspicious, history=history)
        atomic_text(self.raw_dir.parent/"texto_extraido.txt", "\n\f\n".join(p["raw_text"] for p in raw_pages), history=history)
        atomic_text(self.clean_dir.parent/"texto_limpo.txt", "\n\f\n".join(p["clean_text"] for p in pages), history=history)
        report = self.audit(raw_pages, pages, units, changes, suspicious)
        atomic_json(self.report_dir/"relatorio.json", report, history=self.audit_dir/"historico_relatorios")
        atomic_text(self.report_dir/"relatorio.md", self.report_markdown(report), history=history)
        manifest_path = self.root/"manifesto.json"
        manifest = load_json(manifest_path)
        manifest["runs"][self.run_id] = {"extract_key":self.extract_key, "clean_key":self.clean_key,
            "report":str((self.report_dir/"relatorio.json").relative_to(self.root)),
            "segmentation":str(self.segment_dir.relative_to(self.root)), "status":report["status"],
            "runtime":runtime_info(), "checks":report["checks"]}
        manifest["current_run_id"] = self.run_id
        manifest["files"] = {str(p.relative_to(self.root)):sha_file(p) for p in sorted(self.root.rglob("*"))
                            if p.is_file() and p != manifest_path and not any(
                                part.startswith("historico") or part in ("manifestos_anteriores","tentativas")
                                for part in p.relative_to(self.root).parts)}
        atomic_json(manifest_path, manifest, history=self.root/"auditoria"/"manifestos_anteriores")
        return report, units

    def audit(self, raw_pages, pages, units, changes, suspicious):
        review = collections.defaultdict(set)
        errors = []
        for p in raw_pages:
            for flag in p.get("flags", []):
                review[p["page_number"]].add(flag["code"])
            if not p["raw_text"].strip():
                review[p["page_number"]].add("empty_extraction")
            for error in p.get("errors", []):
                errors.append({"page_number":p["page_number"], **error})
        for item in suspicious + units["review_items"]:
            if item.get("page_number"):
                review[item["page_number"]].add(item.get("code", item.get("reason", "segmentation_review")))
        for p in pages:
            raw_size, clean_size = len(p["raw_text"]), len(p["clean_text"])
            if raw_size and (raw_size-clean_size)/raw_size > .10:
                review[p["page_number"]].add("large_content_reduction")
            for flag in p.get("flags", []):
                review[p["page_number"]].add(flag.get("code", "cleaning_review"))
        duplicate = collections.defaultdict(list)
        for p in raw_pages:
            if p["raw_text"].strip():
                duplicate[p["raw_sha256"]].append(p["page_number"])
        duplicate_groups = [v for v in duplicate.values() if len(v)>1]
        for group in duplicate_groups:
            for number in group:
                review[number].add("duplicate_page_text")
        expected = set(range(1, self.page_count+1))
        extracted = {p["page_number"] for p in raw_pages}
        cleaned = {p["page_number"] for p in pages}
        missing = sorted(expected-extracted)
        cleaning_missing = sorted(extracted-cleaned)
        for number in missing + cleaning_missing:
            review[number].add("missing_stage")
            errors.append({"page_number":number, "type":"MissingStage", "message":"Etapa incompleta."})
        restoration = all(validate_clean_record(p) for p in pages)
        segmentation_checks = validate_segmentation(units, pages)
        para_ids = {p["unit_id"] for p in units["paragraphs"]}
        section_ids = {s["unit_id"] for s in units["sections"]}
        chapter_ids = {s["unit_id"] for s in units["chapters"]}
        checks = {
            "original_pdf_sha256_preserved":sha_file(self.original)==self.source_sha==sha_file(self.source),
            "all_pages_extracted":extracted==expected,
            "all_pages_cleaned":cleaned==expected,
            "no_extraction_errors":not errors,
            "cleaning_reversible":restoration,
            "raw_page_hashes":all(p["raw_sha256"]==sha_text(p["raw_text"]) for p in raw_pages),
            "hierarchy":(all(s["parent_paragraph_id"] in para_ids for s in units["sentences"]) and
                all(p.get("parent_section_id") is None or p["parent_section_id"] in section_ids for p in units["paragraphs"]) and
                all(p.get("parent_chapter_id") is None or p["parent_chapter_id"] in chapter_ids for p in units["paragraphs"])),
            "segmentation_validation":segmentation_checks["valid"],
            "no_duplicate_unit_ids":all(len({r["unit_id"] for r in units[k]})==len(units[k])
                                         for k in ("chapters","sections","paragraphs","sentences")),
        }
        # The segmenter validates exact spans; verify again independently against pages.
        by_page = {p["page_number"]:p for p in pages}
        checks["unit_text_offsets"] = all(by_page[u["page_number"]]["clean_text"][u["span"]["start"]:u["span"]["end"]]==u["text"]
            for k in ("paragraphs","sentences") for u in units[k])
        methods = collections.Counter(p["method"] for p in raw_pages)
        kinds = collections.Counter(p.get("inspection", {}).get("page_kind", "erro") for p in raw_pages)
        document_kind = "misto" if kinds.get("digital",0) and (kinds.get("digitalizado",0) or kinds.get("misto",0)) else (
            "digitalizado" if kinds.get("digitalizado",0) and not kinds.get("digital",0) and not kinds.get("misto",0) else (
            "misto" if kinds.get("misto",0) else "digital" if kinds.get("digital",0) else "sem_texto"))
        status = "incompleto" if not all(checks.values()) else "concluido_com_revisao" if review else "concluido_estruturalmente"
        return {"schema_version":"corpus_a.report.v1", "work_id":self.work_id,
            "source_sha256":self.source_sha, "run_id":self.run_id, "status":status,
            "ready_for_vectorization":False, "pdf_kind":document_kind,
            "page_count":self.page_count, "pages_processed":len(raw_pages),
            "direct_pages":methods.get("digital",0), "ocr_pages":methods.get("ocr",0),
            "ocr_failed_pages":methods.get("ocr_error",0), "blank_pages":methods.get("blank",0),
            "ocr_attempted_pages":methods.get("ocr",0)+methods.get("ocr_error",0),
            "page_kinds":dict(kinds), "characters_raw":sum(len(p["raw_text"]) for p in raw_pages),
            "characters_clean":sum(len(p["clean_text"]) for p in pages),
            "changes_count":len(changes), "suspicious_count":len(suspicious),
            "review_pages":[{"page_number":n,"reasons":sorted(r)} for n,r in sorted(review.items())],
            "duplicate_groups":duplicate_groups,"missing_pages":missing,"cleaning_missing_pages":cleaning_missing,
            "errors":errors,"checks":checks,
            "units":{k:len(units[k]) for k in ("chapters","sections","paragraphs","sentences")},
            "limits":["Estrutura não comprova fidelidade editorial ou acerto de OCR.",
                      "Hierarquia e ordem de leitura são candidatas; conferir notas/exemplos e colunas.",
                      "Nenhum embedding ou interpretação foi produzido.",
                      "Revisão humana necessária antes da vetorização."]}

    @staticmethod
    def report_markdown(report):
        lines = ["# Relatório de preparação do Corpus A", "", f"Estado: {report['status']}", "",
                 f"Páginas: {report['pages_processed']}/{report['page_count']}; digital: {report['direct_pages']}; OCR: {report['ocr_pages']}.",
                 f"Caracteres: {report['characters_raw']} → {report['characters_clean']}; alterações: {report['changes_count']}.",
                 "", "## Integridade", ""]
        lines += [f"- {name}: {'PASSOU' if passed else 'FALHOU'}" for name,passed in report["checks"].items()]
        lines += ["", "## Páginas para revisão", ""]
        lines += [f"- Página {p['page_number']}: {', '.join(p['reasons'])}" for p in report["review_pages"]] or ["- Nenhuma indicação automática."]
        lines += ["", "## Limites", ""] + ["- "+x for x in report["limits"]]
        return "\n".join(lines)+"\n"

    def review_page(self, number, dpi=110):
        if not 1 <= number <= self.page_count:
            raise ValueError("Número de página inválido.")
        with fitz.open(self.original) as doc:
            return render_page(doc[number-1], {**self.extraction, "ocr_dpi":dpi})

    def zip_export(self):
        destination = self.root.parent.parent/"exportacoes"
        destination.mkdir(parents=True, exist_ok=True)
        path = destination/f"{self.slug}-{self.source_sha[:12]}-{self.run_id}.zip"
        temp = path.with_suffix(".zip.part")
        with zipfile.ZipFile(temp,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:
            for file in sorted(self.root.rglob("*")):
                if file.is_file():
                    z.write(file, str(Path("corpus_linguistico")/"obras"/self.slug/file.relative_to(self.root)))
        with zipfile.ZipFile(temp) as z:
            if z.testzip() is not None:
                raise ValueError("ZIP não passou no CRC.")
        os.replace(temp,path)
        atomic_text(path.with_suffix(".zip.sha256"), sha_file(path)+"  "+path.name+"\n",
                    history=destination/"historico")
        return path
''')

In [ ]:
Core=sys.modules["corpus_a_core"]
Cleaning=sys.modules["corpus_a_cleaning"]
Segmentation=sys.modules["corpus_a_segmentation"]
print("Ambiente:",Core.runtime_info())
print("GPU:",Core.gpu_info())
if not Core.gpu_info()["available"]:
    print("Extração digital disponível. OCR real requer GPU por padrão; "
          "páginas pendentes serão registradas para retomada.")

## Testes antes do livro

Textos e PDFs sintéticos testam preservação, limpeza reversível, hierarquia,
offsets, retomada, erros por página e exportação. **OCR de teste é simulado**:
verifica o fluxo e a assinatura, sem inferência Surya ou GPU.
Uma falha interrompe a célula. Um conjunto de zero testes é recusado.

In [ ]:
bootstrap_module('corpus_a_tests_cleaning', r'''
"""Contrastive tests for faithful, audited source preparation."""
import copy
import json
import unittest

from corpus_a_cleaning import (CleaningIntegrityError, OCRReviewRequired, apply_patches,
                      clean_page_blocks, detect_repeated_margin_candidates,
                      map_clean_span_to_raw, map_raw_span_to_clean, restore_block,
                      validate_clean_record)


def page(text, **kwargs):
    block = {"block_id": "b1", "raw_text": text, "bbox": [40, 100, 560, 200],
             "role_hint": "prose", "order": 0}
    result = {"page_id": "p1", "page_number": 1, "width": 600, "height": 800,
              "method": "digital", "blocks": [block], "raw_text": text}
    result.update(kwargs)
    return result


class ConservativeCleaningTests(unittest.TestCase):
    def test_default_preserves_linguistic_content_and_unicode(self):
        original = "(1) *Ele não foi.\n(2) ?Ela foi?\n#Não?  Sim!\n\té e\u0301 ℓ ≠ l;\u00a0 não\u202f portanto\nanti-\ninflamatório"
        record = clean_page_blocks(page(original))
        self.assertEqual(record["clean_text"], original)
        self.assertEqual(record["cleaning_summary"]["patch_count"], 0)
        self.assertIn("ambiguous_hard_hyphen_preserved", [f["code"] for f in record["blocks"][0]["flags"]])
        self.assertEqual(restore_block(record["blocks"][0]), original)

    def test_crlf_changes_are_reversible_unicode_offsets(self):
        original = "é😀\r\nnão\r\n"
        block = clean_page_blocks(page(original))["blocks"][0]
        self.assertEqual(block["clean_text"], "é😀\nnão\n")
        self.assertEqual(block["patches"][-1]["raw_start"], 2)
        self.assertEqual(block["patches"][-1]["before"], "\r\n")
        self.assertEqual(restore_block(block), original)
        mapped = map_clean_span_to_raw(block, 2, 3)
        self.assertEqual(mapped["raw_text"], "\r\n")
        self.assertFalse(mapped["exact"])

    def test_hard_hyphen_requires_exact_approved_word(self):
        text = "linguís-\ntica e guarda-\nchuva"
        untouched = clean_page_blocks(page(text))
        self.assertEqual(untouched["clean_text"], text)
        fixed = clean_page_blocks(page(text), {"approved_dehyphenated_words": ["linguística"]})
        self.assertEqual(fixed["clean_text"], "linguística e guarda-\nchuva")
        self.assertEqual(restore_block(fixed["blocks"][0]), text)
        self.assertEqual(fixed["blocks"][0]["patches"][0]["reason"], "explicitly_approved_hard_hyphen_recomposition")

    def test_examples_remain_protected_despite_geometry_and_whitelist(self):
        text = "(1) *Ele não linguís-\ntica"
        source = page(text)
        source["blocks"][0]["line_records"] = [
            {"text": "(1) *Ele não linguís-", "bbox": [40, 100, 280, 112], "font_size": 12},
            {"text": "tica", "bbox": [40, 114, 280, 126], "font_size": 12}]
        cleaned = clean_page_blocks(source, {"approved_dehyphenated_words": ["linguística"], "join_prose_lines": True})
        self.assertEqual(cleaned["clean_text"], text)

    def test_soft_hyphen_opt_in_and_restoration(self):
        text = "coerên\u00ad\ncia e lingüística"
        self.assertEqual(clean_page_blocks(page(text))["clean_text"], text)
        result = clean_page_blocks(page(text), {"join_soft_hyphenated_lines": True})
        self.assertEqual(result["clean_text"], "coerência e lingüística")
        self.assertEqual(restore_block(result["blocks"][0]), text)

    def test_soft_hyphen_marker_deletion_is_distinct_from_line_join(self):
        text = "coerên\u00ad\ncia"
        result = clean_page_blocks(page(text), {"remove_soft_hyphen": True})
        self.assertEqual(result["clean_text"], "coerên\ncia")
        mapped = map_raw_span_to_clean(result["blocks"][0], 6, 7)
        self.assertEqual(mapped["clean_start"], mapped["clean_end"])
        self.assertFalse(mapped["exact"])

    def test_prose_join_requires_opt_in_and_exact_consistent_geometry(self):
        text = "A língua constrói\nsentidos, mas não os determina."
        source = page(text)
        source["blocks"][0]["line_records"] = [
            {"text": "A língua constrói", "bbox": [40, 100, 280, 112], "font_size": 12},
            {"text": "sentidos, mas não os determina.", "bbox": [40, 114, 280, 126], "font_size": 12}]
        self.assertEqual(clean_page_blocks(source)["clean_text"], text)
        cleaned = clean_page_blocks(source, {"join_prose_lines": True})
        self.assertEqual(cleaned["clean_text"], "A língua constrói sentidos, mas não os determina.")
        self.assertEqual(restore_block(cleaned["blocks"][0]), text)
        source["blocks"][0]["line_records"][1]["bbox"][0] = 55
        self.assertEqual(clean_page_blocks(source, {"join_prose_lines": True})["clean_text"], text)

    def test_line_join_refuses_missing_or_nonmatching_metadata(self):
        source = page("A língua constrói\nsentidos.")
        source["blocks"][0]["lines"] = ["A língua constrói", "sentidos."]
        result = clean_page_blocks(source, {"join_prose_lines": True})
        self.assertEqual(result["clean_text"], source["raw_text"])
        self.assertTrue(result["review_required"])

    def test_sentence_punctuation_is_not_erased_by_join(self):
        text = "Não.\nportanto, preserve-se a oposição."
        source = page(text)
        source["blocks"][0]["line_records"] = [
            {"text": "Não.", "bbox": [40, 100, 280, 112], "font_size": 12},
            {"text": "portanto, preserve-se a oposição.", "bbox": [40, 114, 280, 126], "font_size": 12}]
        self.assertEqual(clean_page_blocks(source, {"join_prose_lines": True})["clean_text"], text)

    def test_margin_detection_uses_frequency_and_geometry_no_auto_removal(self):
        pages = []
        for n in range(1, 5):
            source = page("KOCH — TEORIA", page_id=f"p{n}", page_number=n)
            source["blocks"][0]["bbox"] = [40, 10, 280, 30]
            source["blocks"].append({"block_id": "b2", "raw_text": "KOCH — TEORIA", "bbox": [40, 120, 280, 160]})
            source["raw_text"] = "KOCH — TEORIA\n\nKOCH — TEORIA"
            pages.append(source)
        candidates = detect_repeated_margin_candidates(pages)
        self.assertEqual(len(candidates), 1)
        self.assertEqual(candidates[0]["page_count"], 4)
        result = clean_page_blocks(pages[0], repeated_margin_candidates=candidates)
        self.assertEqual(result["clean_text"], pages[0]["raw_text"])
        result = clean_page_blocks(pages[0], {"approved_margin_candidate_ids": [candidates[0]["candidate_id"]]}, candidates)
        self.assertEqual(result["clean_text"], "\n\nKOCH — TEORIA")
        self.assertEqual(result["blocks"][0]["clean_text"], "")
        self.assertEqual(result["blocks"][1]["clean_text"], "KOCH — TEORIA")
        self.assertEqual(restore_block(result["blocks"][0]), "KOCH — TEORIA")

    def test_stale_review_cannot_delete_different_source(self):
        sources = [page("TEORIA", page_id=f"p{n}", page_number=n) for n in range(1, 4)]
        for source in sources:
            source["blocks"][0]["bbox"] = [40, 10, 280, 30]
        candidates = detect_repeated_margin_candidates(sources)
        changed = copy.deepcopy(sources[0])
        changed["raw_text"] = changed["blocks"][0]["raw_text"] = "TEORÍA"
        result = clean_page_blocks(changed, {"approved_margin_candidate_ids": [candidates[0]["candidate_id"]]}, candidates)
        self.assertEqual(result["clean_text"], "TEORÍA")

    def test_manual_margin_text_removal_does_not_delete_body(self):
        source = page("123")
        source["blocks"][0]["bbox"] = [40, 750, 80, 770]
        header = clean_page_blocks(source, {"approved_margin_texts": ["123"]})
        self.assertEqual(header["clean_text"], "")
        body = clean_page_blocks(page("123"), {"approved_margin_texts": ["123"]})
        self.assertEqual(body["clean_text"], "123")

    def test_low_ocr_is_marked_not_corrected_even_with_opt_in(self):
        source = page("coerên\u00ad\ncia\r\n�", method="surya_ocr", ocr_confidence=0.30)
        result = clean_page_blocks(source, {"join_soft_hyphenated_lines": True, "remove_soft_hyphen": True})
        self.assertEqual(result["clean_text"], "coerên\u00ad\ncia\n�")
        self.assertTrue(result["review_required"])
        self.assertIn("ocr_confidence_below_threshold", [f["code"] for f in result["blocks"][0]["flags"]])
        with self.assertRaises(OCRReviewRequired):
            clean_page_blocks(source, {"fail_on_low_ocr_confidence": True})
        with self.assertRaises(CleaningIntegrityError):
            clean_page_blocks(page("texto", method="ocr", ocr_confidence=90))

    def test_absent_ocr_confidence_requires_review(self):
        result = clean_page_blocks(page("A\u00adB", method="ocr"), {"remove_soft_hyphen": True})
        self.assertEqual(result["clean_text"], "A\u00adB")
        self.assertTrue(result["review_required"])

    def test_explicit_replacements_keep_exact_provenance_and_page_offsets(self):
        source = page("ℓ é e\u0301  ")
        source["blocks"].append({"block_id": "b2", "raw_text": "A\r\nB", "bbox": [40, 300, 500, 350]})
        source["raw_text"] += "\n\nA\r\nB"
        result = clean_page_blocks(source, {"explicit_replacements": [{"before": "ℓ", "after": "l", "reason": "reviewed glyph in this extraction"}]})
        self.assertEqual(result["clean_text"], "l é e\u0301  \n\nA\nB")
        first = map_clean_span_to_raw(result["blocks"][0], 0, 1)
        self.assertEqual(first["raw_text"], "ℓ")
        self.assertFalse(first["exact"])
        second = result["blocks"][1]
        self.assertEqual(second["patches"][0]["raw_page_start"], source["raw_text"].index("\r\n"))
        self.assertTrue(validate_clean_record(result))

    def test_sequential_overlapping_edits_restore_original_and_cover_map(self):
        text = "ab\r\ncd"
        cfg = {"explicit_replacements": [
            {"before": "ab", "after": "XYZ", "reason": "reviewed example"},
            {"before": "Y", "after": "", "reason": "reviewed removal"},
            {"before": "Z\nc", "after": "q", "reason": "reviewed crossing edit"}]}
        result = clean_page_blocks(page(text), cfg)
        self.assertEqual(result["clean_text"], "Xqd")
        self.assertEqual(restore_block(result["blocks"][0]), text)
        self.assertTrue(validate_clean_record(result))

    def test_tampering_and_raw_page_mismatch_fail(self):
        record = clean_page_blocks(page("A\r\nB"))
        broken = copy.deepcopy(record["blocks"][0]["patches"])
        broken[0]["before"] = "\n\n"
        with self.assertRaises(CleaningIntegrityError):
            apply_patches("A\r\nB", broken)
        record["clean_blocks"][0]["clean_text"] += "!"
        with self.assertRaises(CleaningIntegrityError):
            validate_clean_record(record)
        source = page("A")
        source["raw_text"] = "B"
        with self.assertRaises(CleaningIntegrityError):
            clean_page_blocks(source)

    def test_source_unchanged_repeated_run_idempotent_json_serializable(self):
        source = page("A\r\nB  é\u00ad")
        source_before = copy.deepcopy(source)
        config = {"remove_soft_hyphen": True}
        result = clean_page_blocks(source, config)
        self.assertEqual(source, source_before)
        self.assertEqual(clean_page_blocks(result, config), result)
        self.assertEqual(json.loads(json.dumps(result, ensure_ascii=False)), result)

    def test_invalid_config_not_silently_accepted(self):
        for config in ({"unicode_normalize": "NFKC"}, {"remove_soft_hyphen": "false"},
                       {"line_left_tolerance": -1}, {"margin_fraction": 3},
                       {"explicit_replacements": [{"before": "", "after": "x", "reason": "bad"}]}):
            with self.subTest(config=config), self.assertRaises(CleaningIntegrityError):
                clean_page_blocks(page("text"), config)


if __name__ == "__main__":
    unittest.main(verbosity=2)
''')

In [ ]:
bootstrap_module('corpus_a_tests_segmentation', r'''
"""Testes de integridade: offsets, cobertura, determinismo e limites estruturais."""

import copy
import unittest

from corpus_a_segmentation import segment_pages, validate_segmentation


SOURCE_SHA256 = "a" * 64


def page_fixture(number, blocks):
    clean_blocks = []
    texts = []
    cursor = 0
    for index, (text, role, font) in enumerate(blocks):
        if index:
            cursor += 2
        clean_blocks.append({"block_id": f"p{number}-b{index + 1}",
                             "clean_text": text, "role_hint": role,
                             "font_size": font, "bbox": [40, index * 70, 500, index * 70 + 40],
                             "clean_page_start": cursor, "clean_page_end": cursor + len(text)})
        cursor += len(text)
        texts.append(text)
    return {"page_number": number, "page_id": f"page-{number}",
            "clean_text": "\n\n".join(texts), "raw_text": "\n\n".join(texts),
            "clean_blocks": clean_blocks, "audit_status": "reviewed"}


class SegmentationTests(unittest.TestCase):
    def assert_integrity(self, result, pages):
        report = validate_segmentation(result, pages)
        self.assertTrue(report["valid"], report["errors"])
        self.assertFalse(report["editorial_structure_validated"])

    def test_exact_unicode_offsets_and_no_normalization(self):
        pages = [page_fixture(1, [("  Ação e cafe\u0301. Outra ideia!  ", "body", 11)])]
        result = segment_pages(pages, "obra", SOURCE_SHA256)
        self.assert_integrity(result, pages)
        self.assertEqual([unit["text"] for unit in result["sentences"]],
                         ["Ação e cafe\u0301.", "Outra ideia!"])
        self.assertEqual(result["paragraphs"][0]["span"]["start"], 2)

    def test_determinism_and_input_not_mutated(self):
        pages = [page_fixture(1, [("Capítulo I", "heading", 18),
                                  ("Primeira frase. Segunda frase.", "body", 11)])]
        before = copy.deepcopy(pages)
        a = segment_pages(pages, "obra", SOURCE_SHA256)
        b = segment_pages(copy.deepcopy(pages), "obra", SOURCE_SHA256)
        self.assertEqual(a, b)
        self.assertEqual(pages, before)
        self.assertNotEqual(a["paragraphs"][0]["unit_id"],
                            segment_pages(pages, "outra-obra", SOURCE_SHA256)["paragraphs"][0]["unit_id"])
        self.assertNotEqual(a["paragraphs"][0]["unit_id"],
                            segment_pages(pages, "obra", "b" * 64)["paragraphs"][0]["unit_id"])

    def test_same_offsets_different_cleaned_text_have_different_ids(self):
        original = [page_fixture(1, [("ABC original.", "body", 11)])]
        changed = [page_fixture(1, [("XYZ original.", "body", 11)])]
        before = segment_pages(original, "obra", SOURCE_SHA256)
        after = segment_pages(changed, "obra", SOURCE_SHA256)
        for unit_type in ("paragraphs", "sentences"):
            self.assertEqual(before[unit_type][0]["span"], after[unit_type][0]["span"])
            self.assertEqual(before[unit_type][0]["block_span"], after[unit_type][0]["block_span"])
            self.assertNotEqual(before[unit_type][0]["unit_id"], after[unit_type][0]["unit_id"])
            self.assertNotEqual(before[unit_type][0]["text_sha256"], after[unit_type][0]["text_sha256"])
        self.assert_integrity(before, original)
        self.assert_integrity(after, changed)

    def test_no_headings_does_not_invent_editorial_structure(self):
        pages = [page_fixture(1, [("O pai entrou. Eu esperei.", "body", 11)])]
        result = segment_pages(pages, "obra", SOURCE_SHA256)
        self.assert_integrity(result, pages)
        self.assertEqual(result["chapters"], [])
        self.assertEqual(result["sections"], [])
        self.assertIsNone(result["paragraphs"][0]["parent_chapter_id"])
        self.assertEqual(result["editorial_structure_status"], "candidate_requires_review")

    def test_chapter_section_hierarchy_across_pages(self):
        page1 = page_fixture(1, [("Capítulo 1 — Estrutura", "heading", 18),
                                 ("1.1 Unidade", "heading", 14),
                                 ("Texto inicial.", "body", 11)])
        page2 = page_fixture(2, [("Continuação do texto.", "body", 11),
                                 ("Capítulo 2 — Outro", "heading", 18),
                                 ("Novo texto.", "body", 11)])
        result = segment_pages([page2, page1], "obra", SOURCE_SHA256)
        self.assert_integrity(result, [page1, page2])
        first, second = result["chapters"]
        section = result["sections"][0]
        continuation = next(p for p in result["paragraphs"] if p["text"] == "Continuação do texto.")
        self.assertEqual(continuation["parent_chapter_id"], first["unit_id"])
        self.assertEqual(continuation["parent_section_id"], section["unit_id"])
        self.assertEqual(first["end_page_number"], 2)
        self.assertNotIn(continuation["unit_id"], second["paragraph_ids"])
        self.assertTrue(all(ch["requires_review"] for ch in result["chapters"]))

    def test_toc_supports_candidates_without_fabricating_absent_titles(self):
        pages = [page_fixture(1, [("Introdução", "body", 11),
                                  ("Detalhes", "body", 11),
                                  ("Notas", "body", 11),
                                  ("Uma frase.", "body", 11)])]
        toc = [[1, "Introdução", 1], [2, "Detalhes", 1], [3, "Notas", 1], [1, "Ausente", 2]]
        result = segment_pages(pages, "obra", SOURCE_SHA256, toc=toc)
        self.assert_integrity(result, pages)
        self.assertEqual(len(result["chapters"]), 1)
        self.assertEqual(len(result["sections"]), 2)
        self.assertEqual(result["sections"][1]["parent_section_id"], result["sections"][0]["unit_id"])
        self.assertTrue(any(item["reason"] == "toc_entry_not_matched_to_extracted_heading"
                            for item in result["review_items"]))
        self.assertFalse(any(ch["text"] == "Ausente" for ch in result["chapters"]))

    def test_note_reference_and_example_are_preserved(self):
        pages = [page_fixture(1, [("Uma observação no corpo.", "body", 11),
                                  ("1. Nota editorial. Não é removida.", "note", 9),
                                  ("1. Koch et al. (2020). Referência.", "reference", 10),
                                  ("Exemplo: Eu esperei.", "example", 10)])]
        result = segment_pages(pages, "obra", SOURCE_SHA256)
        self.assert_integrity(result, pages)
        self.assertEqual({p["role_hint"] for p in result["paragraphs"]},
                         {"body", "note", "reference", "example"})
        self.assertEqual(result["paragraphs"][2]["text"], "1. Koch et al. (2020). Referência.")

    def test_reference_number_and_example_font_do_not_open_sections(self):
        pages = [page_fixture(1, [("Corpo normal.", "body", 11),
                                  ("1.1 Referência numerada.", "reference", 10),
                                  ("Capítulo 5 citado neste exemplo.", "example", 20),
                                  ("Capítulo 8 mencionado em nota.", "note", 9)])]
        result = segment_pages(pages, "obra", SOURCE_SHA256)
        self.assert_integrity(result, pages)
        self.assertEqual(result["chapters"], [])
        self.assertEqual(result["sections"], [])

    def test_two_geometric_hints_only_propose_reviewable_heading(self):
        page = page_fixture(1, [("Corpo anterior.", "body", 11),
                                ("Título inferido", "unknown", 11),
                                ("Corpo posterior.", "body", 11)])
        page["clean_blocks"][0]["bbox"] = [40, 0, 500, 10]
        page["clean_blocks"][1]["bbox"] = [200, 30, 340, 40]
        page["clean_blocks"][2]["bbox"] = [40, 60, 500, 70]
        result = segment_pages([page], "obra", SOURCE_SHA256)
        self.assert_integrity(result, [page])
        chapter = result["chapters"][0]
        self.assertEqual(chapter["structure_status"], "candidate")
        self.assertIn("short_centered_geometry", chapter["heading_evidence"]["evidence_rules"])
        self.assertIn("vertically_isolated_geometry", chapter["heading_evidence"]["evidence_rules"])

    def test_abbreviations_decimals_and_bibliography_number_are_not_cut(self):
        text = "1. A Dra. Koch et al. usam, p.ex., 3.14 unidades. Outra frase."
        pages = [page_fixture(1, [(text, "reference", 11)])]
        result = segment_pages(pages, "obra", SOURCE_SHA256)
        self.assert_integrity(result, pages)
        self.assertEqual([s["text"] for s in result["sentences"]],
                         ["1. A Dra. Koch et al. usam, p.ex., 3.14 unidades.", "Outra frase."])

    def test_abbreviation_at_possible_end_is_marked_uncertain(self):
        pages = [page_fixture(1, [("Itens etc. Depois ele saiu.", "body", 11)])]
        result = segment_pages(pages, "obra", SOURCE_SHA256)
        self.assert_integrity(result, pages)
        self.assertEqual(len(result["sentences"]), 1)
        self.assertTrue(result["sentences"][0]["split_uncertain"])
        self.assertEqual(result["sentences"][0]["sentence_boundary_flags"][0]["reason"],
                         "abbreviation_may_end_sentence")

    def test_quotes_and_ellipses_keep_exact_text_and_flag_uncertainty(self):
        text = 'Ele disse: “Olá.” Depois hesitou... Talvez voltasse. “Sim!”, disse ela.'
        pages = [page_fixture(1, [(text, "body", 11)])]
        result = segment_pages(pages, "obra", SOURCE_SHA256)
        self.assert_integrity(result, pages)
        self.assertEqual(result["sentences"][0]["text"], 'Ele disse: “Olá.”')
        self.assertIn("ellipsis_boundary_undetermined",
                      {flag["reason"] for flag in result["paragraphs"][0]["sentence_boundary_flags"]})
        self.assertTrue(any("“Sim!”, disse ela." == sent["text"] for sent in result["sentences"]))

    def test_multiple_paragraphs_and_simple_line_wraps(self):
        pages = [page_fixture(1, [("Primeira linha\ncontinua aqui.\n\nSegundo parágrafo.", "body", 11)])]
        result = segment_pages(pages, "obra", SOURCE_SHA256)
        self.assert_integrity(result, pages)
        self.assertEqual(len(result["paragraphs"]), 2)
        self.assertEqual(result["paragraphs"][0]["text"], "Primeira linha\ncontinua aqui.")

    def test_invalid_offsets_fall_back_without_loss_or_duplication(self):
        pages = [page_fixture(1, [("Texto original.", "body", 11), ("Segundo texto.", "note", 9)])]
        pages[0]["clean_blocks"][1]["clean_page_start"] = 0
        result = segment_pages(pages, "obra", SOURCE_SHA256)
        self.assert_integrity(result, pages)
        self.assertEqual([p["text"] for p in result["paragraphs"]], ["Texto original.", "Segundo texto."])
        self.assertTrue(all(p["alignment_basis"] == "page_fallback_after_mapping_error"
                            for p in result["paragraphs"]))
        self.assertTrue(any(item["reason"] == "block_page_offsets_do_not_match"
                            for item in result["review_items"]))

    def test_text_outside_blocks_preserved(self):
        page = page_fixture(1, [("Bloco presente.", "body", 11)])
        page["clean_text"] += "\n\nTexto extra."
        result = segment_pages([page], "obra", SOURCE_SHA256)
        self.assert_integrity(result, [page])
        self.assertEqual(result["paragraphs"][-1]["text"], "Texto extra.")
        self.assertFalse(result["paragraphs"][-1]["source_ref"]["source_block"])

    def test_sequential_alignment_handles_repeated_block_text(self):
        pages = [page_fixture(1, [("Texto igual.", "body", 11), ("Texto igual.", "body", 11)])]
        for block in pages[0]["clean_blocks"]:
            del block["clean_page_start"]
            del block["clean_page_end"]
        result = segment_pages(pages, "obra", SOURCE_SHA256)
        self.assert_integrity(result, pages)
        self.assertNotEqual(result["paragraphs"][0]["unit_id"], result["paragraphs"][1]["unit_id"])
        self.assertLess(result["paragraphs"][0]["span"]["start"], result["paragraphs"][1]["span"]["start"])

    def test_missing_blocks_uses_whole_page_without_title_claim(self):
        pages = [{"page_number": 1, "page_id": "p1", "clean_text": "Corpo completo. Outra frase."}]
        result = segment_pages(pages, "obra", SOURCE_SHA256)
        self.assert_integrity(result, pages)
        self.assertEqual(len(result["paragraphs"]), 1)
        self.assertEqual(len(result["sentences"]), 2)

    def test_invalid_page_identifiers_are_rejected(self):
        page = page_fixture(1, [("Texto.", "body", 11)])
        with self.assertRaises(ValueError):
            segment_pages([page, copy.deepcopy(page)], "obra", SOURCE_SHA256)
        page["page_number"] = 0
        with self.assertRaises(ValueError):
            segment_pages([page], "obra", SOURCE_SHA256)

    def test_validator_detects_modified_excerpt(self):
        pages = [page_fixture(1, [("Trecho original.", "body", 11)])]
        result = segment_pages(pages, "obra", SOURCE_SHA256)
        result["sentences"][0]["text"] = "Trecho inventado."
        report = validate_segmentation(result, pages)
        self.assertFalse(report["valid"])
        self.assertTrue(any(error.startswith("page_span_mismatch:") for error in report["errors"]))

    def test_no_interpretative_fields_or_theoretical_labels(self):
        pages = [page_fixture(1, [("Meu pai me corrigiu. Eu não senti raiva.", "body", 11)])]
        result = segment_pages(pages, "obra", SOURCE_SHA256)
        forbidden_keys = {"interpretation", "hypothesis", "w", "ro", "rs", "diagnosis", "semantic_class"}

        def check_keys(value):
            if isinstance(value, dict):
                self.assertFalse(forbidden_keys.intersection(value))
                for nested in value.values():
                    check_keys(nested)
            elif isinstance(value, list):
                for nested in value:
                    check_keys(nested)

        check_keys(result)


if __name__ == "__main__":
    unittest.main()
''')

In [ ]:
bootstrap_module('corpus_a_tests_core', r'''
"""Functional tests: synthetic PDFs only; Surya inference is explicitly simulated."""
import copy
import importlib.util
import json
from pathlib import Path
import tempfile
import unittest
from unittest.mock import patch

import fitz
from PIL import Image
import corpus_a_core as core


def make_pdf(path, *, scan=False, blank=False, rotated=False, columns=False):
    doc = fitz.open()
    for n in range(1, 4):
        page = doc.new_page(width=595, height=842)
        page.insert_text((45, 27), "CABECALHO DE TESTE", fontsize=8)
        if n == 1:
            page.insert_text((45, 72), "CAPITULO 1 - Coerencia", fontsize=17)
        page.insert_textbox(fitz.Rect(45, 115, 550, 190),
            "A coerencia depende do contexto. Nao removemos negacoes nem conectivos.\n"
            "Por exemplo, repeticoes podem ser significativas.", fontsize=11)
        page.insert_text((45, 220), "(1) * Os livro chegou.", fontsize=11)
        page.insert_text((45, 248), "(2) Nao e o mesmo que sim.", fontsize=11)
        page.insert_text((45, 800), "1 Nota: preservar esta referencia.", fontsize=8)
        if n == 3 and rotated:
            page.set_rotation(90)
    if columns:
        page = doc.new_page(width=595, height=842)
        for n in range(4):
            page.insert_text((30, 100+n*40), f"Coluna esquerda {n}.",fontsize=10)
            page.insert_text((315, 100+n*40), f"Coluna direita {n}.",fontsize=10)
    if scan:
        fixture = fitz.open()
        p = fixture.new_page(width=595,height=842)
        p.insert_text((50, 120), "Texto digitalizado de exemplo.",fontsize=15)
        pix = p.get_pixmap(matrix=fitz.Matrix(1.4,1.4))
        page = doc.new_page(width=595,height=842)
        page.insert_image(page.rect, stream=pix.tobytes("png"))
        fixture.close()
    if blank:
        doc.new_page(width=595,height=842)
    doc.save(path)
    doc.close()
    return Path(path)


class SimulatedOCR:
    """A controlled fixture adapter, NOT a Surya/GPU validation."""
    def __init__(self, fail=False):
        self.calls = 0
        self.fail = fail
    def recognize(self, image):
        self.calls += 1
        if self.fail:
            raise RuntimeError("Falha OCR simulada para testar retomada.")
        return [{"text":"Texto digitalizado de exemplo.",
                 "confidence":0.99, "bbox_image":[20,40,image.width-20,65],
                 "polygon_image":[[20,40],[image.width-20,40],[image.width-20,65],[20,65]]}]
    def layout(self, image):
        return [{"label":"Text","position":0,"confidence":0.98,
                 "bbox_image":[0,0,image.width,image.height]}]
    def release(self):
        pass


class PipelineTests(unittest.TestCase):
    def setUp(self):
        self.temp = tempfile.TemporaryDirectory()
        self.base = Path(self.temp.name)
        self.output = self.base/"corpus_linguistico"
        self.bib = {"slug":"teste","titulo":"Livro sintetico","autores":["Teste"],"edicao":"fixture"}
        self.progress = patch.object(core, "tqdm", lambda items, **kwargs: items)
        self.progress.start()
    def tearDown(self):
        self.progress.stop()
        self.temp.cleanup()
    def pipeline(self, **kwargs):
        pdf = make_pdf(self.base/"teste.pdf", **kwargs)
        return core.CorpusPipeline(pdf,self.output,self.bib,adapter=SimulatedOCR())

    def test_digital_full_pipeline_integrity_offsets_and_zip(self):
        p = self.pipeline()
        source = core.sha_file(p.source)
        self.assertEqual(p.extract()["ok"],3)
        pages,candidates = p.clean()
        self.assertTrue(any(c["signature"]=="CABECALHO DE TESTE" for c in candidates))
        self.assertIn("CABECALHO DE TESTE",pages[0]["clean_text"])
        report,units = p.export(pages)
        self.assertTrue(all(report["checks"].values()),report["checks"])
        self.assertEqual(report["direct_pages"],3)
        self.assertEqual(report["ocr_pages"],0)
        self.assertFalse(report["ready_for_vectorization"])
        self.assertIn("(1) * Os livro chegou.",pages[0]["clean_text"])
        self.assertTrue(units["paragraphs"])
        self.assertTrue(units["sentences"])
        self.assertEqual(core.sha_file(p.original),source)
        self.assertTrue(p.zip_export().is_file())

    def test_incremental_resume_does_not_reextract_or_overwrite_raw(self):
        p = self.pipeline()
        p.extract(page_numbers=[1])
        old = p.raw_path(1).read_bytes()
        self.assertFalse(p.raw_path(2).exists())
        summary = p.extract()
        self.assertEqual(summary["reused"],1)
        self.assertEqual(summary["ok"],2)
        self.assertEqual(p.raw_path(1).read_bytes(),old)
        pages,_ = p.clean()
        first = p.clean_path(1).read_bytes()
        again,_ = p.clean()
        self.assertEqual(first,p.clean_path(1).read_bytes())
        self.assertEqual(pages,again)

    def test_repeat_export_is_idempotent(self):
        p=self.pipeline();p.extract();pages,_=p.clean()
        p.export(pages)
        before=(p.root/"manifesto.json").read_bytes()
        p.export(pages)
        self.assertEqual(before,(p.root/"manifesto.json").read_bytes())

    def test_ocr_failure_checkpoint_and_successful_retry(self):
        p = self.pipeline(scan=True)
        failing = SimulatedOCR(fail=True)
        p.adapter = failing
        p.extract()
        pages,_ = p.clean()
        report,_ = p.export(pages)
        self.assertEqual(report["status"],"incompleto")
        self.assertEqual(report["ocr_failed_pages"],1)
        self.assertFalse(report["checks"]["no_extraction_errors"])
        p.adapter = SimulatedOCR()
        summary = p.extract(retry_errors=True)
        self.assertEqual(summary["reused"],3)
        self.assertEqual(p.adapter.calls,1)
        pages,_ = p.clean()
        report,_ = p.export(pages)
        self.assertEqual(report["ocr_pages"],1)
        self.assertEqual(report["ocr_failed_pages"],0)
        self.assertTrue(all(report["checks"].values()))
        self.assertTrue(list((p.audit_dir/"tentativas").glob("*")))

    def test_blank_page_is_reported_not_discarded(self):
        p = self.pipeline(blank=True)
        p.extract(); pages,_=p.clean(); report,_=p.export(pages)
        self.assertEqual(report["page_count"],4)
        self.assertEqual(len(pages),4)
        self.assertEqual(report["blank_pages"],1)
        self.assertIn(4,[r["page_number"] for r in report["review_pages"]])

    def test_same_slug_different_pdf_refused(self):
        p = self.pipeline()
        other = make_pdf(self.base/"outro.pdf",blank=True)
        with self.assertRaisesRegex(ValueError,"outro PDF"):
            core.CorpusPipeline(other,self.output,self.bib,adapter=SimulatedOCR())
        self.assertEqual(core.sha_file(p.original),p.source_sha)

    def test_raw_geometry_tampering_is_detected(self):
        p=self.pipeline(); p.extract([1])
        value=core.load_json(p.raw_path(1)); value["blocks"][0]["bbox"][0]+=1
        p.raw_path(1).write_text(json.dumps(value),encoding="utf-8")
        with self.assertRaisesRegex(ValueError,"Checkpoint divergente"):
            p.extract([1])

    def test_partial_run_report_is_incomplete(self):
        p=self.pipeline(); p.extract([1]); pages,_=p.clean(); report,_=p.export(pages)
        self.assertEqual(report["status"],"incompleto")
        self.assertEqual(report["missing_pages"],[2,3])

    def test_clean_config_versions_do_not_overwrite_previous(self):
        p=self.pipeline();p.extract();pages,candidates=p.clean();p.export(pages)
        original=p.clean_path(1).read_bytes()
        q=core.CorpusPipeline(p.source,self.output,self.bib,adapter=SimulatedOCR(),
            cleaning={"approved_margin_candidate_ids":[next(c["candidate_id"] for c in candidates if c["signature"]=="CABECALHO DE TESTE")]})
        self.assertEqual(p.extract_key,q.extract_key)
        self.assertNotEqual(p.clean_key,q.clean_key)
        q.extract();clean,_=q.clean(); report,_=q.export(clean)
        self.assertNotIn("CABECALHO DE TESTE",clean[0]["clean_text"])
        self.assertEqual(p.clean_path(1).read_bytes(),original)
        self.assertGreater(report["changes_count"],0)
        self.assertTrue(report["checks"]["cleaning_reversible"])

    def test_multicolumn_is_flagged_and_content_preserved(self):
        p=self.pipeline(columns=True);p.extract();pages,_=p.clean();report,_=p.export(pages)
        review={r["page_number"]:r["reasons"] for r in report["review_pages"]}
        self.assertIn("possible_multiple_columns",review[4])
        for n in range(4):
            self.assertIn(f"Coluna esquerda {n}.",pages[3]["clean_text"])
            self.assertIn(f"Coluna direita {n}.",pages[3]["clean_text"])

    def test_rotated_page_retains_text_and_records_coordinate_space(self):
        p=self.pipeline(rotated=True);p.extract();raw=p.read_raw()
        self.assertEqual(raw[-1]["rotation"],90)
        self.assertIn("Nao removemos",raw[-1]["raw_text"])
        for b in raw[-1]["blocks"]:
            self.assertTrue(0<=b["bbox"][0]<=raw[-1]["width"])
            self.assertTrue(0<=b["bbox"][1]<=raw[-1]["height"])

    def test_native_short_page_is_not_forced_to_ocr_without_image(self):
        doc=fitz.open();page=doc.new_page();page.insert_text((60,80),"Titulo");doc.save(self.base/"curto.pdf");doc.close()
        p=core.CorpusPipeline(self.base/"curto.pdf",self.output,self.bib,adapter=SimulatedOCR())
        p.extract();self.assertEqual(p.adapter.calls,0)
        self.assertEqual(p.read_raw()[0]["method"],"digital")

    def test_stale_clean_pages_cannot_be_exported_after_ocr_retry(self):
        p=self.pipeline(scan=True);p.adapter=SimulatedOCR(fail=True)
        p.extract();old_pages,_=p.clean()
        p.adapter=SimulatedOCR();p.extract()
        with self.assertRaisesRegex(ValueError,"Extração/limpeza divergentes"):
            p.export(old_pages)

    def test_margin_candidate_change_invalidates_partial_clean_cache(self):
        p=self.pipeline();p.extract([1,2]);pages,_=p.clean()
        old=pages[0]["margin_candidates_sha256"]
        p.extract();new,_=p.clean()
        self.assertNotEqual(old,new[0]["margin_candidates_sha256"])

    def test_clean_without_extraction_detects_raw_tampering(self):
        p=self.pipeline();p.extract([1])
        value=core.load_json(p.raw_path(1));value["width"]+=1
        p.raw_path(1).write_text(json.dumps(value),encoding="utf-8")
        with self.assertRaisesRegex(ValueError,"Checkpoint divergente"):
            p.clean()

    def test_real_surya_adapter_uses_verified_signature_with_repeat_preservation(self):
        # No Surya library/model is installed: call arguments are verified via controlled predictor.
        adapter=core.SuryaAdapter(self.base)
        class Line:
            text="nao nao"; confidence=.9; bbox=[0,0,10,10];polygon=[[0,0],[10,0],[10,10],[0,10]]
        class Result:
            text_lines=[Line()]
        calls=[]
        def predictor(images,**kwargs):
            calls.append(kwargs);return [Result()]
        adapter.recognizer=predictor;adapter.detector=object()
        with patch.object(adapter,"_prepare",return_value="cpu"):
            image=Image.new("RGB",(20,20));result=adapter.recognize(image);image.close()
        self.assertEqual(result[0]["text"],"nao nao")
        self.assertFalse(calls[0]["drop_repeated_text"])
        self.assertFalse(calls[0]["sort_lines"])
        self.assertEqual(calls[0]["recognition_batch_size"],1)


if __name__=="__main__":
    unittest.main()
''')

In [ ]:
import unittest
test_modules=[sys.modules[name] for name in
              ("corpus_a_tests_cleaning","corpus_a_tests_segmentation","corpus_a_tests_core")]
suite=unittest.TestSuite(unittest.defaultTestLoader.loadTestsFromModule(m) for m in test_modules)
TEST_RESULT=unittest.TextTestRunner(verbosity=1).run(suite)
if TEST_RESULT.testsRun==0 or not TEST_RESULT.wasSuccessful():
    raise RuntimeError("Testes falharam: diagnostique antes do livro.")
print("Testes concluídos:",TEST_RESULT.testsRun)

## Demonstração textual

A regra padrão preserva hífen ambíguo. A segunda saída demonstra uma palavra
explicitamente aprovada. Exemplos gramaticais ficam protegidos.
A lista de aprovações do livro começa vazia.

In [ ]:
import json
DEMO_BLOCKS=[
 {"block_id":"demo:b1","role_hint":"body","bbox":[30,100,500,140],
  "raw_text":"Não é indiferente: porém, a coerên-\ncia depende do contexto.\r\n"
             "Repetições: não, não. Acento decomposto: cafe\u0301."},
 {"block_id":"demo:b2","role_hint":"example","bbox":[30,160,500,200],
  "raw_text":"(1) * Os livro chegou.\n(2) ? Uma palavra parti-\nda como objeto de análise."}
]
DEMO={"page_id":"demo:p1","page_number":1,"width":595,"height":842,"method":"digital",
      "blocks":DEMO_BLOCKS,"raw_text":"\n\n".join(b["raw_text"] for b in DEMO_BLOCKS)}
padrao=Cleaning.clean_page_blocks(DEMO)
aprovado=Cleaning.clean_page_blocks(DEMO,{"approved_dehyphenated_words":["coerência"]})
print("EXTRAÍDO ORIGINALMENTE:\n",DEMO["raw_text"])
print("\nPADRÃO:\n",padrao["clean_text"])
print("\nCOM APROVAÇÃO DA PALAVRA:\n",aprovado["clean_text"])
for b in aprovado["clean_blocks"]:
    for edit in b["patches"]:
        print(json.dumps({"block_id":b["block_id"],**edit},ensure_ascii=False))
    assert Cleaning.restore_block(b)==b["raw_text"]
assert "(1) * Os livro chegou." in aprovado["clean_text"]
assert "não, não" in aprovado["clean_text"]
assert "cafe\u0301" in aprovado["clean_text"]
print("Restauração e preservação conferidas.")

## Etapa A — entrada e preservação

O original é copiado, identificado por SHA256 e preservado integralmente.
A página física do PDF usa base 1. A numeração impressa exige conferência.

In [ ]:
if IN_COLAB and (ARMAZENAR_NO_DRIVE or INPUT_MODE=="drive"):
    from google.colab import drive
    drive.mount("/content/drive")

test_pdf=os.environ.get("CORPUS_A_TEST_PDF")
if IN_COLAB:
    OUTPUT_BASE=Path(PASTA_DRIVE) if ARMAZENAR_NO_DRIVE else Path("/content/corpus_linguistico")
else:
    OUTPUT_BASE=Path(os.environ.get("CORPUS_A_TEST_OUTPUT",
                                   str(Path(tempfile.gettempdir())/"corpus_linguistico")))
OUTPUT_BASE.mkdir(parents=True,exist_ok=True)
cached_manifest=OUTPUT_BASE/"obras"/SLUG/"manifesto.json"
if test_pdf:
    PDF_ENTRADA=Path(test_pdf)
elif RETOMAR_ORIGINAL_SALVO and cached_manifest.exists() and not CAMINHO_PDF:
    old=Core.load_json(cached_manifest)
    PDF_ENTRADA=cached_manifest.parent/old["original"]
    print("Retomando original:",PDF_ENTRADA)
elif INPUT_MODE=="upload":
    if not IN_COLAB:
        raise RuntimeError("No modo local use path/CAMINHO_PDF; no Colab, upload.")
    from google.colab import files
    uploaded=files.upload()
    if len(uploaded)!=1: raise ValueError("Selecione exatamente um PDF.")
    filename,content=next(iter(uploaded.items()))
    if not filename.lower().endswith(".pdf"): raise ValueError("Selecione um PDF.")
    input_dir=Path("/content/corpus_a_entrada")
    input_dir.mkdir(parents=True,exist_ok=True)
    PDF_ENTRADA=input_dir/(hashlib.sha256(content).hexdigest()+".pdf")
    if PDF_ENTRADA.exists():
        if PDF_ENTRADA.read_bytes()!=content: raise ValueError("Entrada divergente.")
    else:
        PDF_ENTRADA.write_bytes(content)
    del uploaded,content
else:
    if not CAMINHO_PDF: raise ValueError("Preencha CAMINHO_PDF.")
    PDF_ENTRADA=Path(CAMINHO_PDF)

EXTRACAO_CONFIG=dict(Core.EXTRACTION_DEFAULTS)
EXTRACAO_CONFIG.update({"ocr_dpi":180,"use_surya_layout":False,
                       "force_ocr_pages":[],"force_direct_pages":[],
                       "save_review_images":False})
pipeline=Core.CorpusPipeline(PDF_ENTRADA,OUTPUT_BASE,METADADOS,
    extraction=EXTRACAO_CONFIG,cache_dir=OUTPUT_BASE/"cache_surya")
print("Obra:",pipeline.work_id,"\nPáginas:",pipeline.page_count)
print("SHA256:",pipeline.source_sha,"\nPasta:",pipeline.root)

## Etapa B — diagnóstico do PDF

Combina texto digital, qualidade de caracteres, imagens e conteúdo vetorial.
Páginas digitais usam extração direta; OCR é recomendado para texto inacessível,
corrompido ou páginas digitalizadas. Imagens também podem conter exemplos, figuras
ou tabelas: essas regiões são registradas para revisão.
Use force_ocr_pages/force_direct_pages após conferir as páginas.

In [ ]:
from collections import Counter
DIAGNOSTICO=[]
with Core.fitz.open(pipeline.original) as document:
    for page in Core.tqdm(document,desc="Diagnóstico do PDF"):
        native_blocks=Core.digital_blocks(page)
        info=Core.inspect_page(page,native_blocks,pipeline.extraction)
        DIAGNOSTICO.append({"page_number":page.number+1,**info})
Core.atomic_json(pipeline.audit_dir/"diagnostico_pdf.json",DIAGNOSTICO,
                 history=pipeline.audit_dir/"historico")
print("Tipos:",dict(Counter(x["page_kind"] for x in DIAGNOSTICO)))
print("OCR recomendado:",[x["page_number"] for x in DIAGNOSTICO if x["needs_ocr"]])

## Etapa C — extração incremental

Surya 0.17.0 foi escolhido por API examinada, não por ser a versão mais recente.
O notebook fixa Transformers 4.56.2, verifica Torch >=2.7,<3 e aproveita o Torch CUDA
do Colab. Pesos são obtidos sob demanda pela biblioteca oficial.

Reconhecimento e detecção recebem uma página. Repetições são preservadas com
drop_repeated_text=False. Layout é opcional e usa outro modelo Foundation;
o notebook libera modelos anteriores antes de carregar o próximo.

Ordem geométrica e posições de layout são candidatas. Colunas, tabelas, notas,
equações e exemplos podem exigir revisão editorial.

In [ ]:
PAGINAS_A_PROCESSAR=[]  # vazio=todas; por exemplo [1,2,3] para o primeiro piloto
REPROCESSAR_FALHAS=True
EXTRACTION_SUMMARY=pipeline.extract(PAGINAS_A_PROCESSAR or None,
                                   retry_errors=REPROCESSAR_FALHAS)
print("Esta execução:",EXTRACTION_SUMMARY)
RAW_PAGES=pipeline.read_raw()
print("Métodos:",dict(Counter(p["method"] for p in RAW_PAGES)))
print("Falhas:",[{"pagina":p["page_number"],"erros":p["errors"]}
                for p in RAW_PAGES if p["status"]!="ok"])

### Conferir uma página

Renderiza somente a página escolhida. Compare imagem, notas, exemplos e blocos.
A cópia original e o texto bruto ficam disponíveis mesmo após erro de OCR.

In [ ]:
PAGINA_PARA_CONFERIR=1 #@param {type:"integer"}
from IPython.display import display
image=pipeline.review_page(PAGINA_PARA_CONFERIR)
try: display(image)
finally: image.close()
raw=next((p for p in RAW_PAGES if p["page_number"]==PAGINA_PARA_CONFERIR),None)
if raw:
    print("Método:",raw["method"],"| Ordem:",raw.get("reading_order"))
    for b in raw["blocks"]:
        print("\nBLOCO:",b["block_id"],b["role_hint"],b["bbox"])
        print(b["raw_text"][:1200])

## Etapa D — limpeza conservadora

Candidatos de cabeçalho/rodapé usam frequência e geometria. Notas repetidas
também podem aparecer: confira cada ocorrência antes de aprovar.
Nada é removido por padrão. Hífens, soft hyphen e junção de linhas exigem opções
explícitas; recomposição de palavra exige lista aprovada.
Ruído, corrupção e espaçamento incertos são encaminhados à revisão.

In [ ]:
MARGIN_CANDIDATES=Cleaning.detect_repeated_margin_candidates(RAW_PAGES)
for c in MARGIN_CANDIDATES:
    print(c["candidate_id"],c["side"],repr(c["signature"]),
          "páginas:",[o["page_number"] for o in c["occurrences"]])
print("Candidatos para conferir:",len(MARGIN_CANDIDATES))

In [ ]:
CABECALHOS_RODAPES_APROVADOS=[]  # IDs, após revisão das ocorrências
PALAVRAS_RECOMPOSTAS_APROVADAS=[]  # palavras completas conferidas no original
SUBSTITUICOES_EXPLICITAS=[]  # [{"before":"...", "after":"...", "reason":"conferido na página"}]
REMOVER_SOFT_HYPHEN=False #@param {type:"boolean"}
JUNTAR_SOFT_HYPHEN_LINHA=False #@param {type:"boolean"}
JUNTAR_LINHAS_EM_PROSA=False #@param {type:"boolean"}
CLEAN_CONFIG=dict(Cleaning.DEFAULT_CLEAN_CONFIG)
CLEAN_CONFIG.update({
 "approved_margin_candidate_ids":CABECALHOS_RODAPES_APROVADOS,
 "approved_dehyphenated_words":PALAVRAS_RECOMPOSTAS_APROVADAS,
 "explicit_replacements":SUBSTITUICOES_EXPLICITAS,
 "remove_soft_hyphen":REMOVER_SOFT_HYPHEN,
 "join_soft_hyphenated_lines":JUNTAR_SOFT_HYPHEN_LINHA,
 "join_prose_lines":JUNTAR_LINHAS_EM_PROSA})
pipeline.adapter.release()
pipeline=Core.CorpusPipeline(PDF_ENTRADA,OUTPUT_BASE,METADADOS,
    extraction=EXTRACAO_CONFIG,cleaning=CLEAN_CONFIG,cache_dir=OUTPUT_BASE/"cache_surya")
CLEAN_PAGES,MARGIN_CANDIDATES=pipeline.clean()
print("Páginas limpas:",len(CLEAN_PAGES))
print("Alterações:",sum(p["cleaning_summary"]["patch_count"] for p in CLEAN_PAGES))
print("Chave da limpeza:",pipeline.clean_key)

## Etapa E — três representações e auditoria

Cada bloco mantém raw_text, clean_text, hashes, patches sequenciais e span_map.
Offsets são pontos de código Unicode, base zero e fim exclusivo. Mudanças registram
página, bloco, motivo e texto antes/depois. Versões anteriores permanecem preservadas.

In [ ]:
ALL_CHANGES=[]
for p in CLEAN_PAGES:
    for b in p["clean_blocks"]:
        assert Cleaning.restore_block(b)==b["raw_text"]
        ALL_CHANGES.extend({"page_number":p["page_number"],"block_id":b["block_id"],**e}
                           for e in b["patches"])
print("Restauração conferida.")
print(json.dumps(ALL_CHANGES[:12],ensure_ascii=False,indent=2))
if len(ALL_CHANGES)>12: print("Todas as alterações serão exportadas em alteracoes.jsonl.")

## Etapas F e G — segmentação e relatório

Unidades preservam texto exato, IDs estáveis, página, bloco, hierarquia e metadados.
Títulos, capítulos e seções são candidatos apoiados por tipografia, geometria,
rótulos e sumário, quando disponível. Estrutura ausente não é inventada.
Notas, referências e exemplos permanecem no corpus. Fronteiras incertas são marcadas.

Estado incompleto indica falha ou etapa ausente. Concluído com revisão indica
integridade estrutural e pendências. ready_for_vectorization permanece False:
a preparação não aprova editorialmente a obra.

In [ ]:
REPORT,UNITS=pipeline.export(CLEAN_PAGES)
print(json.dumps(REPORT,ensure_ascii=False,indent=2))
from IPython.display import Markdown
display(Markdown((pipeline.report_dir/"relatorio.md").read_text(encoding="utf-8")))

In [ ]:
for p in UNITS["paragraphs"][:8]:
    print("\nID:",p["unit_id"],"| página:",p["page_number"],
          "| papel:",p["role_hint"],"| posição:",p["span"])
    print(p["text"][:250])
print("Unidades:",{k:len(UNITS[k]) for k in ("chapters","sections","paragraphs","sentences")})

## Etapa H — exportação

    corpus_linguistico/
        obras/
            coerencia_textual/
                original/
                extraido/execucoes/
                limpo/execucoes/
                segmentado/execucoes/
                auditoria/execucoes/
                relatorios/
                manifesto.json

TXT bruto/limpo por página e por obra; JSONL de páginas, capítulos, seções,
parágrafos e períodos; alterações e suspeitas em JSONL; relatório JSON/Markdown.
Manifesto aponta para a execução atual e registra hashes e versões anteriores.
ZIP inclui a obra e o original, sem o cache dos modelos. CRC e SHA256 são conferidos.

In [ ]:
EXPORT_PATH=pipeline.zip_export()
print("ZIP:",EXPORT_PATH)
print("Tamanho MiB:",round(EXPORT_PATH.stat().st_size/2**20,2))
print("SHA256:",Core.sha_file(EXPORT_PATH))
print("Manifesto:",pipeline.root/"manifesto.json")
DOWNLOAD_ZIP_AGORA=False #@param {type:"boolean"}
if DOWNLOAD_ZIP_AGORA and IN_COLAB:
    from google.colab import files
    files.download(str(EXPORT_PATH))
else:
    print("No Colab, habilite DOWNLOAD_ZIP_AGORA ou baixe pelo painel de arquivos/Drive.")

## Retomada e limites das verificações

- Reexecute em ordem com mesmo PDF, slug e metadados.
- Erros de GPU/download/OCR são registrados por página. Após corrigir a causa,
  retome extração e limpeza; sucessos anteriores são reutilizados.
- Confira páginas suspeitas no original. Aprovações e configurações geram outra
  versão de limpeza, sem substituição silenciosa da anterior.
- Colunas, tabelas, imagens de exemplos, fórmulas e fontes especiais podem demandar
  revisão. Confiança OCR não é probabilidade calibrada.
- Integridade estrutural não demonstra fidelidade editorial.

**Validação da entrega:** código e notebook testáveis localmente com PDFs sintéticos;
API da distribuição oficial Surya 0.17.0 inspecionada.
**Não houve execução no Google Colab nem inferência real Surya em GPU.**
Nenhum PDF do livro foi fornecido. No Colab, faça primeiro um piloto de poucas páginas.

Fontes da API:
- [PyPI 0.17.0](https://pypi.org/project/surya-ocr/0.17.0/)
- [Código oficial v0.17.0](https://github.com/datalab-to/surya/tree/v0.17.0)
- [RecognitionPredictor](https://github.com/datalab-to/surya/blob/v0.17.0/surya/recognition/__init__.py)
- [LayoutPredictor](https://github.com/datalab-to/surya/blob/v0.17.0/surya/layout/__init__.py)
- [PyMuPDF](https://pymupdf.readthedocs.io/)

Surya baixa pesos de models.datalab.to pela biblioteca oficial, com TLS padrão.
O notebook não desativa verificações de transporte. Modelos são carregados sob demanda.